# Algorithm for stock selection

In [3]:
#Import the required libraries
import pandas as pd
import requests as r
import statistics as s
import os.path
from os import path

In [312]:
pip install tqdm

Note: you may need to restart the kernel to use updated packages.


In [35]:
#Create a list of stocks that can be used to test the functioning of the algorithm
#before running it on the whole SP500
trial_list=['MMM','ACC','BRK.A','AAPL']
SP500Symbols=pd.read_excel(r'C:\Users\filip\Desktop\SP500_List.xlsx')
SP500=SP500Symbols['Symbol']

In [237]:
#Create a function to get the earnings per share of a stock of choice
def get_NI_pershare(symbol,year):
    #call the api and retrieve the keymetrics json
    s_metrics=r.get("https://financialmodelingprep.com/api/v3/company-key-metrics/"+symbol+'?apikey=REDACTED')
    #transform the json in a python object, a dictionary
    if s_metrics.status_code==200:
        s_metrics_dic=s_metrics.json()
        #get the information needed from the dictionary, and transform it into a float
        #Some symbols do not have any data. I check it with bool
        if(bool(s_metrics_dic)):
            if len(s_metrics_dic['metrics'])>=(year+1):
                if(s_metrics_dic['metrics'][year]['Net Income per Share']):
                    return float(s_metrics_dic['metrics'][year]['Net Income per Share'])
                else:
                    return 0
            else:
                return 0
        else:
            return 0
    else:
        return 0

In [239]:
def get_PE(symbol,year):
    #call the api and retrieve the keymetrics json
    s_metrics=r.get("https://financialmodelingprep.com/api/v3/company-key-metrics/"+symbol+'?apikey=REDACTED')
    #transform the json in a python object, a dictionary
    if s_metrics.status_code==200:
        s_metrics_dic=s_metrics.json()
    #get the information needed from the dictionary, and transform it into a float
    #Some symbols do not have any data. I check it with bool
        if(bool(s_metrics_dic)):
            if len(s_metrics_dic['metrics'])>=(year+1):
                if(s_metrics_dic['metrics'][year]['PE ratio']):
                    return float(s_metrics_dic['metrics'][year]['PE ratio'])
                else:
                    return 0
            else:
                return 0
        else:
            return 0
    else:
        return 0

In [240]:
def get_PB(symbol,year):
    #call the api and retrieve the keymetrics json
    s_metrics=r.get("https://financialmodelingprep.com/api/v3/company-key-metrics/"+symbol+'?apikey=REDACTED')
    #transform the json in a python object, a dictionary
    if s_metrics.status_code==200:
        s_metrics_dic=s_metrics.json()
    #get the information needed from the dictionary, and transform it into a float
    #Some symbols do not have any data. I check it with bool
        if(bool(s_metrics_dic)):
            if len(s_metrics_dic['metrics'])>=(year+1):
                if(s_metrics_dic['metrics'][year]['PB ratio']):
                    return float(s_metrics_dic['metrics'][year]['PB ratio'])
                else:
                    return 0
            else:
                return 0
        else:
            return 0
    else:
        return 0

In [241]:
#Create a function to get the current ratio of a stock of choice
def get_current_ratio(symbol,year):
    #call the api and retrieve the keymetrics json
    s_metrics=r.get("https://financialmodelingprep.com/api/v3/company-key-metrics/"+symbol+'?apikey=REDACTED')
    #transform the json in a python object, a dictionary
    if s_metrics.status_code==200:
        s_metrics_dic=s_metrics.json()
    #get the information needed from the dictionary, and transform it into a float
    #Some symbols do not have any data. I check it with bool
        if(bool(s_metrics_dic)):
            if len(s_metrics_dic['metrics'])>=(year+1):
                if(s_metrics_dic['metrics'][year]['Current ratio']):
                    return float(s_metrics_dic['metrics'][year]['Current ratio'])
                else:
                    return 0
            else:
                return 0
        else:
            return 0
    else:
        return 0

In [242]:
#Create a function to get the PB ratio of a stock of choice
def get_book_pershare(symbol,year):
    #call the api and retrieve the keymetrics json
    s_metrics=r.get("https://financialmodelingprep.com/api/v3/company-key-metrics/"+symbol+'?apikey=REDACTED')
    #transform the json in a python object, a dictionary
    if s_metrics.status_code==200:
        s_metrics_dic=s_metrics.json()
    #get the information needed from the dictionary, and transform it into a float
    #Some symbols do not have any data. I check it with bool
        if(bool(s_metrics_dic)):
            if len(s_metrics_dic['metrics'])>=(year+1):
                if(s_metrics_dic['metrics'][year]['Book Value per Share']):
                    return float(s_metrics_dic['metrics'][year]['Book Value per Share'])
                else:
                    return 0
            else:
                return 0
        else:
            return 0
    else:
        return 0

In [244]:
#Create a function that checks the liquidity of a company
def check_liquidity(symbol,year):
    #call the api and retrieve the keymetrics json
    s_financials=r.get("https://financialmodelingprep.com/api/v3/financials/balance-sheet-statement/"+symbol+'?apikey=REDACTED')
    #transform the json in a python object, a dictionary
    if s_financials.status_code==200:
        s_financials_dic=s_financials.json()
    #get the information needed from the dictionary, and transform it into a float
    #Some symbols do not have any data. I check it with bool
        if(bool(s_financials_dic)):
            if (bool(s_financials_dic['financials'])):
                if len(s_financials_dic['financials'])>=(year+1):
                    if(s_financials_dic['financials'][year]['Long-term debt']):
                        Long_term_debt=float(s_financials_dic['financials'][year]['Long-term debt'])
                        if(s_financials_dic['financials'][year]['Total current assets']):
                            Total_current_assets=float(s_financials_dic['financials'][year]['Total current assets'])
                            if(s_financials_dic['financials'][year]['Total current liabilities']):
                                Total_current_liabilities=float(s_financials_dic['financials'][year]['Total current liabilities'])
                                if(Long_term_debt-(Total_current_assets-Total_current_liabilities))>0:
                                    return 'no'
                                else:
                                    return 'ok'
                            else:
                                return 'not available'
                        else:
                            return 'not available'
                    else:
                        return 'not available'
                else:
                    return 'index not found'
            else:
                return 'not available'
        else:
            return 'not available'
    else:
        return 'not available'

In [245]:
#Create a function that returns a company's revenues
def get_revenues(symbol,year):
    #call the api and retrieve the keymetrics json
    s_financials=r.get("https://financialmodelingprep.com/api/v3/financials/income-statement/"+symbol+'?apikey=REDACTED')
    #transform the json in a python object, a dictionary
    if s_financials.status_code==200:
        s_financials_dic=s_financials.json()
    #get the information needed from the dictionary, and transform it into a float
    #Some symbols do not have any data. I check it with bool
        if(bool(s_financials_dic)):
            if len(s_financials_dic['financials'])>=(year+1):
                if(s_financials_dic['financials'][year]['Revenue']):
                    return float(s_financials_dic['financials'][year]['Revenue'])
                else:
                    return 0
            else:
                return 0
        else:
            return 0
    else:
        return 0

In [246]:
#Create a function that returns a company's EPS Growth. This function is much less reliable than the one
#below, as it does not consider a three year average when calculating the growth, and it is still unclear
#where this value are taken from. They do not seem to represent neither the absolute difference in EPS nohr 
#the percentage difference. 
#rangE can be 1,3,5 or 10
def get_EPS_growth(symbol,year,rangE):
    #call the api and retrieve the keymetrics json
    s_financials=r.get("https://financialmodelingprep.com/api/v3/financial-growth/"+symbol+'?apikey=REDACTED')
    #transform the json in a python object, a dictionary
    if s_financials.status_code==200:
        s_financials_dic=s_financials.json()
    #get the information needed from the dictionary, and transform it into a float
    #Some symbols do not have any data. I check it with bool
        if(bool(s_financials_dic)):
            if len(s_financials_dic)>=(year+1):
                if rangE==1:
                    if(s_financials_dic[year]['netIncomeGrowth']):
                        return 100*float(s_financials_dic[year]['netIncomeGrowth'])
                    else:
                        return 0
                elif rangE==3:
                    if(s_financials_dic[year]['threeYNetIncomeGrowthPerShare']):
                        return 100*float(s_financials_dic[year]['threeYNetIncomeGrowthPerShare'])
                    else:
                        return 0
                elif rangE==5:
                    if(s_financials_dic[year]['fiveYNetIncomeGrowthPerShare']):
                        return 100*float(s_financials_dic[year]['fiveYNetIncomeGrowthPerShare'])
                    else:
                        return 0
                elif rangE==10:
                    if(s_financials_dic[year]['tenYNetIncomeGrowthPerShare']):
                        return 100*float(s_financials_dic[year]['tenYNetIncomeGrowthPerShare'])
                    else:
                        return 0            
            else:
                return 0
        else:
            return 0
    else:
        return 0

In [247]:
#Create a function that calculates the earnings growth
def get_earnings_growth(symbol):
    import statistics
    first_three=[]
    last_three=[]
    s_metrics=r.get("https://financialmodelingprep.com/api/v3/company-key-metrics/"+symbol+'?apikey=REDACTED')
    if s_metrics.status_code==200:
        s_metrics=s_metrics.json()
        if(bool(s_metrics)):
            tot_years=len(s_metrics['metrics'])-1
            for i in range(3):
                if(get_NI_pershare(symbol,i)):
                    if(get_NI_pershare(symbol,tot_years-i)):
                        last_three.append(get_NI_pershare(symbol,i))
                        first_three.append(get_NI_pershare(symbol,tot_years-i))
                    else:
                        return('not available')
                else:
                    return('not available')
        
            last_three_avg=statistics.mean(last_three)
            first_three_avg=statistics.mean(first_three)
            return ((last_three_avg-first_three_avg)/abs(first_three_avg)*100)
    
        else:
            return('not available')
    else:
        return ('not available')

In [248]:
#Create a function that checks if the earnings in the previous eleven years were positive
#This condition may be a bit too harsh (e.g. Accenture only has one negative year, in 2009)
#If I notice that most companies had negative earnings in that year I could remove it from the constraints
def positive_earnings(symbol):  
    s_metrics=r.get("https://financialmodelingprep.com/api/v3/company-key-metrics/"+symbol+'?apikey=REDACTED')
    if s_metrics.status_code==200:
        s_metrics_dic=s_metrics.json()
        if(bool(s_metrics_dic)):
            tot_years=len(s_metrics_dic['metrics'])  
            for i in range(tot_years):
                if (get_NI_pershare(symbol,i)<0):
                    return 'no'
                elif(get_NI_pershare(symbol,i)==0):
                    return 'not available'
            return 'ok'    
    else:
        return 'not available'

In [249]:
#Create a function to get the payout ratio of a stock of choice
def get_payout_ratio(symbol,year):
    #call the api and retrieve the keymetrics json
    s_metrics=r.get("https://financialmodelingprep.com/api/v3/company-key-metrics/"+symbol+'?apikey=REDACTED')
    #transform the json in a python object, a dictionary
    if s_metrics.status_code==200:
        s_metrics_dic=s_metrics.json()
    #get the information needed from the dictionary, and transform it into a float
    #Some symbols do not have any data. I check it with bool
        if(bool(s_metrics_dic)):
            if len(s_metrics_dic['metrics'])>=(year+1):
                if(s_metrics_dic['metrics'][year]['Payout Ratio']):
                    return float(s_metrics_dic['metrics'][year]['Payout Ratio'])
                else:
                    return 0
            else:
                return 0
        else:
            return 0
    else:
        return 0

In [251]:
def get_payout_SD(symbol):
    payout_list=[]
    import statistics
    s_metrics=r.get("https://financialmodelingprep.com/api/v3/company-key-metrics/"+symbol+'?apikey=REDACTED')
    if s_metrics.status_code==200:
        s_metrics=s_metrics.json()
        if(bool(s_metrics)):
            tot_years=len(s_metrics['metrics'])
            for i in range(tot_years):
                if(get_payout_ratio(symbol,i)):
                    payout_list.append(get_payout_ratio(symbol,i))
            if len(payout_list)>1:
                return statistics.stdev(payout_list)/statistics.mean(payout_list)*100
            else:
                return ('no payout recorded')
    
        else:
            return('not available') 
    else: 
        return ('not available')

In [252]:
def positive_payout(symbol):  
    s_metrics=r.get("https://financialmodelingprep.com/api/v3/company-key-metrics/"+symbol+'?apikey=REDACTED')
    if s_metrics.status_code==200:
        s_metrics_dic=s_metrics.json()
        if(bool(s_metrics_dic)):
            tot_years=len(s_metrics_dic['metrics'])  
            for i in range(tot_years):
                if (get_payout_ratio(symbol,i)==0):
                    return 'no'
            return 'ok' 
    else:
        return 'not available'
        

In [268]:
def get_net_current_assets(s,year):
    
    ent_values=r.get('https://financialmodelingprep.com/api/v3/enterprise-values/'+s+'?apikey=REDACTED')
    if ent_values.status_code==200:
        ent_values_dic=ent_values.json()
        if (bool(ent_values_dic)):
            tot_years=len(ent_values_dic)
            if tot_years>=(year+1):
                if (bool(ent_values_dic[year]['numberOfShares'])):
                    num_of_shares=ent_values_dic[year]['numberOfShares']
                else:
                    return 0.0
            else:
                return 0.0
        else:
            return 0.0
    else:
        return 0.0
    
    s_financials=r.get("https://financialmodelingprep.com/api/v3/financials/balance-sheet-statement/"+s+'?apikey=REDACTED')
    if s_financials.status_code==200:
        s_financials_dic=s_financials.json()
        if(bool(s_financials_dic)):
            if (bool(s_financials_dic['financials'])):
                if len(s_financials_dic['financials'])>=(year+1):
                    if(s_financials_dic['financials'][year]['Total current assets']):
                        Total_current_assets=float(s_financials_dic['financials'][year]['Total current assets'])
                        if(s_financials_dic['financials'][year]['Total liabilities']):
                            Total_liabilities=float(s_financials_dic['financials'][year]['Total liabilities'])
                            return ((Total_current_assets-Total_liabilities)/num_of_shares)
                        else:
                            return 0.0
                    else:
                        return 0.0
                else:
                    return 0.0
            else:
                return 0.0
        else:
            return 0.0
    else:
        return 0.0

In [1]:
def get_company_name(symbol):
    company_profile=r.get('https://financialmodelingprep.com/api/v3/profile/'+symbol+'?apikey=REDACTED')
    if company_profile.status_code==200:
        company_profile_dic=company_profile.json()
        if (bool(company_profile_dic)):
            if (bool(company_profile_dic[0])):
                return company_profile_dic[0]['companyName']
            else:
                return 'NA'
        else:
            return 'NA'
    else: 
        return 'NA'
    

In [255]:
def get_company_industry(symbol):
    company_profile=r.get('https://financialmodelingprep.com/api/v3/profile/'+symbol+'?apikey=REDACTED')
    if company_profile.status_code==200:
        company_profile_dic=company_profile.json()
        if (bool(company_profile_dic)):
            if (bool(company_profile_dic[0])):
                return company_profile_dic[0]['industry']
            else:
                return 'NA'
        else:
            return 'NA'
    else:
        return 'NA'

In [369]:
from time import sleep

In [403]:
def create_stats_DF(date,year,SP500,months_list):
    if path.exists(r'C:\Users\filip\Downloads\price_cache.csv'):
        temp_stats=pd.read_csv(r'C:\Users\filip\Downloads\price_cache.csv')
        #inserted NYSEandNDQ for the moment being 
        historical_prices=pd.Series(temp_stats.Price.values,index=NYSEandNDQ).to_dict()
    else:
        historical_prices=get_historical_prices(date,temp_stats,months_list)
        price_cache=pd.DataFrame(historical_prices.items(),columns=['Symbol','Price'])
        price_cache=price_cache.set_index('Symbol')
        price_cache.to_csv(r'C:\Users\filip\Downloads\price_cache.csv')
        
    Stats_cache=pd.read_csv(r'C:\Users\filip\Downloads\Stats_cache.csv')
    Stats_cache=Stats_cache.set_index('Unnamed: 0')
    print("Loaded " + str(len(Stats_cache)) + " stats from cache")
    print(Stats_cache.index.to_list())
    SP500_stats=pd.DataFrame()
    for s in tqdm(SP500):
        sleep(3)
        if s in Stats_cache.index:
            print("Found " + s + "in the cache")
            SP500_stats.append(Stats_cache.loc[s,:])
            continue
        else:
            if historical_prices[s]=='no data':
                continue
            else:
                SP500_stats.loc[s,'Price']=historical_prices[s]
                SP500_stats.loc[s,'EPS']=get_NI_pershare(s,year)
                #I assume that when year 0 is selected, the user wants to know the PE based on the present price of the stock(s)
                if year==0:
                        if get_NI_pershare(s,0):
                            SP500_stats.loc[s,'PE']=float(historical_prices[s])/get_NI_pershare(s,0)
                        else:
                            SP500_stats.loc[s,'PE']=0
                        if get_book_pershare(s,0):
                            SP500_stats.loc[s,'PB']=float(historical_prices[s])/get_book_pershare(s,0)
                        else:
                            SP500_stats.loc[s,'PB']=0      
                else:
                    SP500_stats.loc[s,'PE']=get_PE(s,year)
                    SP500_stats.loc[s,'PB']=get_PB(s,year)

                SP500_stats.loc[s,'PExPB']=SP500_stats.loc[s,'PB']*SP500_stats.loc[s,'PE']
                SP500_stats.loc[s,'Current ratio']=get_current_ratio(s,year)
                SP500_stats.loc[s,'Liquidity check']=check_liquidity(s,year)
                SP500_stats.loc[s,'Revenues']=get_revenues(s,year)
                SP500_stats.loc[s,'Earnings Growth %']=get_EPS_growth(s,year,1)
                SP500_stats.loc[s,'Earnings Growth 3y %']=get_EPS_growth(s,year,3)
                SP500_stats.loc[s,'Earnings Growth 5y %']=get_EPS_growth(s,year,5)
                SP500_stats.loc[s,'Earnings Growth 10y %']=get_EPS_growth(s,year,10)
                SP500_stats.loc[s,'Payout ratio']=get_payout_ratio(s,year)
                SP500_stats.loc[s,'Net Current Assets PS']=get_net_current_assets(s,year)

                Stats_cache=Stats_cache.append(SP500_stats.loc[s,:])
                Stats_cache.to_csv(r'C:\Users\filip\Downloads\Stats_cache.csv')
            
    return SP500_stats

In [375]:
SC=pd.read_csv(r'C:\Users\filip\Downloads\Stats_cache.csv')
SC=SC.set_index('Unnamed: 0')

In [378]:
SC=SC.append(SC.loc['VCVCU',:])

In [379]:
SC

,Price,EPS,PE,PB,PExPB,Current ratio,Liquidity check,Revenues,Earnings Growth %,Earnings Growth 3y %,Earnings Growth 5y %,Earnings Growth 10y %,Payout ratio,Net Current Assets PS
Unnamed: 0,,,,,,,,,,,,,,
VCVCU,10.61,0.0,0.0,0.0,0.0,0.0,not available,0.0,0.0,0.0,0.0,0.0,0.0,0.0
VCVCU,10.61,0.0,0.0,0.0,0.0,0.0,not available,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [334]:
Stats_c.to_csv(r'C:\Users\filip\Downloads\Stats_cache.csv')

In [ ]:
Stats_c.to_csv(r')

In [328]:
Stats_c=Stats_c.append(SP500_stats.loc['AAPL',:])

In [332]:
'TSLA' in Stats_c.index

False

In [145]:
SP500_stats.to_csv(r'C:\Users\filip\Downloads\SP500_stats_23_12.csv')

In [103]:
SP500_stats_filtered=SP500_stats[SP500_stats['Revenues']>500000000]

In [105]:
SP500_stats_filtered=SP500_stats_filtered[SP500_stats_filtered['Current ratio']<2]

In [106]:
SP500_stats_filtered=SP500_stats_filtered[SP500_stats_filtered['PB']>0]

In [108]:
SP500_stats_filtered=SP500_stats_filtered[SP500_stats_filtered['PExPB']<22]

In [110]:
SP500_stats_filtered=SP500_stats_filtered[SP500_stats_filtered['EPS']>0]

In [113]:
SP500_stats_filtered=SP500_stats_filtered[SP500_stats_filtered['Earnings Growth %']>0]

In [126]:
Symbols=SP500_stats_filtered.index

In [128]:
for s in Symbols:
    SP500_stats_filtered.loc[s,'Company Name']=get_company_name(s)
    SP500_stats_filtered.loc[s,'Industry']=get_company_industry(s)

C:\Users\filip\Anaconda3\lib\site-packages\pandas\core\indexing.py:376: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self.obj[key] = _infer_fill_value(value)
C:\Users\filip\Anaconda3\lib\site-packages\pandas\core\indexing.py:494: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self.obj[item] = s


In [132]:
for s in Symbols:
    SP500_stats_filtered.loc[s,'Net Current Assets']=get_net_current_assets(s,0)

In [144]:
SP500_stats_filtered.to_csv(r'C:\Users\filip\Downloads\SP500_stats_filtered_23_12.csv')

In [ ]:
%%time
SP500_stats=pd.DataFrame(columns=["Price"])
for s in SP500:
    s_profile=r.get("https://financialmodelingprep.com/api/v3/company/profile/"+ s)
    s_profile_dic=s_profile.json()
    s_profile_DF=pd.DataFrame(s_profile_dic)
    s_profile_DF=s_profile_DF.drop('symbol',axis=1)
    s_profile_DF=s_profile_DF.transpose()
    s_price=int(s_profile_DF['price'])
    SP500_stats.loc[s,'Price']=s_price
    SP500_stats.loc[s,'EPS']=get_NI_pershare(s,0)
    if(get_NI_pershare(s,0)):
        SP500_stats.loc[s,'Current PE']=s_price/get_NI_pershare(s,0)
    else:
        SP500_stats.loc[s,'Current PE']=0
    if(get_book_pershare(s,0)):
        SP500_stats.loc[s,'PB ratio']=s_price/get_book_pershare(s,0)
    else:
        SP500_stats.loc[s,'PB ratio']=0
    SP500_stats.loc[s,'PExPB']=SP500_stats.loc[s,'PB ratio']*SP500_stats.loc[s,'Current PE']
    SP500_stats.loc[s,'Current ratio']=get_current_ratio(s,0)
    SP500_stats.loc[s,'Liquidity check']=check_liquidity(s,0)
    SP500_stats.loc[s,'Revenues']=get_revenues(s,0)
    SP500_stats.loc[s,'Earnings Growth 5y']=get_earnings_growth(s)
    SP500_stats.loc[s,'Positive Earnings 11y']=positive_earnings(s)
    SP500_stats.loc[s,'Payout ratio']=get_payout_ratio(s,0)
    SP500_stats.loc[s,'Paying Dividend 11y']=positive_payout(s)
    SP500_stats.loc[s,'Payout %Dev']=get_payout_SD(s)

In [ ]:
SP500_stats

In [ ]:
SP500_stats.to_csv("SP500_stats.csv")

In [ ]:
SP500_filtered=SP500_stats[SP500_stats['Revenues']>500000000]

In [ ]:
SP500_filtered=SP500_filtered[SP500_filtered['Current ratio']>2]

In [ ]:
SP500_filtered=SP500_filtered[SP500_filtered['Liquidity check']=='ok']

In [ ]:
SP500_filtered=SP500_filtered[SP500_filtered['Positive Earnings 11y']=='ok']

In [ ]:
SP500_filtered=SP500_filtered[SP500_filtered['PExPB']<22] 

In [ ]:
SP500_filtered=SP500_filtered[SP500_filtered['Earnings Growth 5y']>22.5] 

In [ ]:
SP500_filtered

In [ ]:
trial_run=SP500_stats[SP500_stats['Current PE']<5]
trial_run=trial_run[trial_run['Current PE']>0]

In [ ]:
#They would need to be examined in a different way. However, I do not want to lose that information.
rel_stock_with_no_earnings_info=[]
for s in trial_run.index:
    if trial_run.loc[s,'Earnings Growth 5y']=='not available':
        rel_stock_with_no_earnings_info.append(s)
trial_run=trial_run.drop(rel_stock_with_no_earnings_info)


In [ ]:
trial_run[trial_run['Earnings Growth 5y']>0]

# How much to invest in each stock?

In [ ]:
SP500_stats=pd.read_csv('SP500_stats.csv')

In [ ]:
ind=SP500_stats['Unnamed: 0']
SP500_stats.index=ind
#It is not pretty but it worked

In [ ]:
#Filtering the SP500 according to the criteria set by Benjamin Graham
SP500_filtered=SP500_stats[SP500_stats['Revenues']>500000000]

SP500_filtered=SP500_filtered[SP500_filtered['Current ratio']>2]

SP500_filtered=SP500_filtered[SP500_filtered['Liquidity check']=='ok']

SP500_filtered=SP500_filtered[SP500_filtered['Positive Earnings 11y']=='ok']

SP500_filtered=SP500_filtered[SP500_filtered['PExPB']<22] 

SP500_filtered=SP500_filtered.astype({'Earnings Growth 5y': 'float'})

SP500_filtered=SP500_filtered[SP500_filtered['Earnings Growth 5y']>22.5] 

In [25]:
#I need to create a DF with the month (e.g. 1,2,3,..) and the corresponding number of days
months_list=pd.DataFrame(columns=['Month','NumberOfDays'])
for i in range(12):
    months_list.loc[i,'Month']=i+1
for i in range(12):
    if i==0 or i==2 or i==4 or i==6 or i==7 or i==9 or i==11:
        months_list.loc[i,'NumberOfDays']=31
    elif i==3 or i==5 or i==8 or i==10:
        months_list.loc[i,'NumberOfDays']=30
    else:
        months_list.loc[i,'NumberOfDays']=28

In [26]:
months_list

,Month,NumberOfDays
0,1,31
1,2,28
2,3,31
3,4,30
4,5,31
5,6,30
6,7,31
7,8,31
8,9,30
9,10,31


In [77]:
get_corresponding_day('2012-01-01','MMM',months_list)

2130

In [256]:
#date has to be written in the form 'year-month-day'
#when I created this function I did as if it was certain that the user was going to input the last day of one year. But it is not necessarily the case.
def get_corresponding_day(date,s,months_list):
    if s=='^GSPC':
        historical_p=r.get("https://financialmodelingprep.com/api/v3/historical-price-full/%5EGSPC?apikey=REDACTED")
    else:
        historical_p=r.get("https://financialmodelingprep.com/api/v3/historical-price-full/"+s+"?serietype=line&apikey=REDACTED")
    if historical_p.status_code==200:    
        historical_p=historical_p.json()
        if bool(historical_p):
            if bool(historical_p['historical']):
                d=0
                #even though there is a price point, it is highly unlikely that it is the price point I was looking for
                #hence, I select no data, as I do for the no data option
                if len(historical_p['historical'])==1:
                    return 0
                for i in range(len(historical_p['historical'])):
                    #if I leave it like this it will repeat the thing below
                    if historical_p['historical'][i]['date']==date:
                        d=i
                while(not bool(d)): 
                    date=list(date)
                    temp1=date[8]
                    temp2=date[9]
                    #Does the number of days only have one digit
                    if temp1=='0':
                        #Is the number of days different than 01
                        if temp2!='1':
                            day=int(date[9])
                            ndays=day-1
                            date[9]=str(ndays)
                        elif temp2=='1': 
                        #the day numer is 01
                            temp3=date[5]
                            temp4=date[6]
                            month=temp3+temp4
                        #the month number is 01         
                            if temp3=='0' and temp4=='1':
                                year="".join(date[:4])
                                year=int(year)
                                year=year-1
                                nyear=list(str(year))
                                date[0]=nyear[0]
                                date[1]=nyear[1]
                                date[2]=nyear[2]
                                date[3]=nyear[3]
                                date[5]='1'
                                date[6]='2'
                                date[8]='3'
                                date[9]='1'
                       #the month number is not 01               
                            else:
                                if temp3=='1' and temp4=='0':
                                    m=9
                                    date[5]='0'
                                    date[6]='9'

                                elif temp3=='0':
                                    m=int(temp4)-1
                                    nmonth=str(m)
                                    date[6]=nmonth
                                else:
                                    m=int(month)-1
                                    nmonth=list(str(m))
                                    date[5]=nmonth[0]
                                    date[6]=nmonth[1]
                                #I have to go back by one month, which means that I need the number of days from the month before the present
                                #one
                                ndays=int(months_list[months_list['Month']==m]['NumberOfDays'])
                                ndays_list=list(str(ndays))
                                date[8]=ndays_list[0]
                                date[9]=ndays_list[1]

                    elif temp1=='1' and temp2=='0':
                        date[8]='0'
                        date[9]='9'

                    else:    
                        day=int(temp1+temp2)
                        ndays=day-1
                        ndays_list=list(str(ndays))
                        date[8]=ndays_list[0]
                        date[9]=ndays_list[1]

                    date="".join(date)
                    for i in range(len(historical_p['historical'])):
                        if historical_p['historical'][i]['date']==date:
                            if bool(i):
                                d=i
                return d            


            else:
                return 0
        else:
            return 0
    else:
        return 0



In [257]:
def get_corresponding_date(starting_date,day_interval,s,months_list):
    historical_p=r.get("https://financialmodelingprep.com/api/v3/historical-price-full/"+s+"?serietype=line&apikey=REDACTED")
    if historical_p.status_code==200: 
        historical_p=historical_p.json()
        starting_day=get_corresponding_day(starting_date,s,months_list)
        final_day=starting_day-day_interval
        return (historical_p['historical'][final_day]['date'])
    else:
        return 0   

In [258]:
def get_historical_prices(date,stats,months_list):
    stocks_symbols=[]
    historical_prices=[]
    for s in stats.index:
        day=get_corresponding_day(date,s,months_list)
        historical_price=r.get("https://financialmodelingprep.com/api/v3/historical-price-full/"+s+"?serietype=line&apikey=REDACTED")
        if historical_price.status_code==200: 
            historical_price=historical_price.json()
            stocks_symbols.append(s)
            if (day):
                historical_prices.append(historical_price['historical'][day]['close'])
            else:
                historical_prices.append('no data')
        else:
            historical_prices.append('no data')
    return dict(zip(stocks_symbols, historical_prices))

In [ ]:
get_historical_prices("2009-12-31",test_portfolio_2009)

In [ ]:
#SP500_filtered.loc['JPM',]
#for i in 
get_historical_prices('2018-09-20',SP500_filtered)

In [30]:
def how_much_per_share(capital_invested,date,stats,months_list):
    historical_prices=get_historical_prices(date,stats,months_list)
    number_of_issues=len(stats['Price'])
    capital_invested_per_stock=capital_invested/number_of_issues
    stocks_symbols=[]
    how_much_perstock=[]
    for s in stats.index:
        stocks_symbols.append(s)
        how_much_perstock.append(capital_invested_per_stock/historical_prices[s])
    return dict(zip(stocks_symbols, how_much_perstock))

In [ ]:
how_much_per_share(200,'2009--20',SP500_filtered)

In [31]:
def get_todays_prices(stats):
    stocks_symbols=[]
    todays_prices=[]
    for s in stats.index:
        today_price=r.get("https://financialmodelingprep.com/api/v3/quote-short/"+s+'?apikey=REDACTED')
        today_price=today_price.json()
        stocks_symbols.append(s)
        todays_prices.append(today_price['price'])
    return dict(zip(stocks_symbols, todays_prices))


In [ ]:
get_todays_prices(SP500_filtered)

In [25]:
def get_aggregated_portfolio_growth(capital_invested,starting_date,end_date,stats):
    portfolio=how_much_per_share(capital_invested,starting_date,stats)
    historical_prices=get_historical_prices(starting_date,stats)
    if end_date=='today':
        todays_prices=get_todays_prices(stats)
    else:
        todays_prices=get_historical_prices(end_date,stats)
    past_value=0
    present_value=0
    for s in stats.index:
        past_value+=portfolio[s]*historical_prices[s]
        present_value+=portfolio[s]*todays_prices[s]
    return (present_value-past_value)/past_value*100

In [ ]:
get_aggregated_portfolio_growth(200,'2018-09-20',SP500_filtered)

In [49]:
#annual compound is considered
#How do I get the yearly compound?
#the main issue is that the count of each stock is different, depending on when it went public
#it is actually not a problem, because, even if the count is different, the distance between 
#the two dates will be the same
def get_yearly_portfolio_growth(capital_invested,starting_date,end_date,stats,months_list):
    compounded_growth=get_aggregated_portfolio_growth(capital_invested,starting_date,end_date,stats)
    starting_day=get_corresponding_day(starting_date,'MMM',months_list)
    final_day=get_corresponding_day(end_date,'MMM',months_list)
    #get the historical prices of a random stock to see how many days have been recorded in total
    #historical_p=r.get("https://financialmodelingprep.com/api/v3/historical-price-full/"+'MMM'+"?serietype=line")
    #historical_p=historical_p.json()
    #today=len(historical_p['historical'])
    #now I calculated the number of years between today and the starting day
    years=(starting_day-final_day)/365
    return ((1+compounded_growth/100)**(1/years)-1)*100

In [ ]:
get_yearly_portfolio_growth(200,'2008-03-06',SP500_filtered)

In [110]:
sp_historic=pd.read_csv(r"C:\Users\filip\Downloads\^GSPC (1).csv")

In [111]:
#First you need to download the csv file with historical data from Yahoo Finance
def get_SP500_growth(starting_date,end_date,historic_prices):
    starting_price=float(historic_prices[historic_prices['Date']==starting_date]['Close'])
    final_price=float(historic_prices[historic_prices['Date']==end_date]['Close'])
    return (final_price-starting_price)/starting_price*100

In [112]:
def get_SP500_yearly_growth(starting_date,end_date,historic_prices):
    compounded_growth=get_SP500_growth(starting_date,end_date,historic_prices)
    days=(sp_historic[sp_historic['Date']==end_date]['Close'].index-sp_historic[sp_historic['Date']==starting_date]['Close'].index)[0]
    years=days/365
    return ((1+compounded_growth/100)**(1/years)-1)*100

In [ ]:
get_SP500_growth('2009-12-31','2019-12-31',sp_historic)

In [ ]:
get_SP500_yearly_growth('2009-12-31','2019-12-31',sp_historic)

# What if my portfolio is updated every year based on the changes in the stock that are selected by the algorithm?

In [28]:
def updated_portfolio_with_rebalancing(capital_invested,starting_date,review_date,filtering_1,filtering_2):
    #to be dropped and #to_be_added are only needed if the final goal is to automatize the trading itself.
    #in that way, degiro will know which ones it has to drop, and which ones it has to buy.
    to_be_dropped=[]
    to_be_added=[]
    starting_portfolio=how_much_per_share(capital_invested,starting_date,filtering_1)
    prices_on_review_date=get_historical_prices(review_date,filtering_1)
    present_value_portfolio=0
    for i in filtering_1.index: 
        present_value_portfolio+=starting_portfolio[i]*prices_on_review_date[i]
        if i not in filtering_2.index:
            to_be_dropped.append(i)
    for j in filtering_2.index:
        if j not in filtering_1.index:
             to_be_added.append(j)
    new_portfolio=how_much_per_share(present_value_portfolio,review_date,filtering_2)
    return new_portfolio,to_be_dropped, to_be_added

In [29]:
def portfolio_growth_with_rebalancing(capital_invested,starting_date,review_date,final_date,test_portfolio_1,test_portfolio_2):
    starting_portfolio=how_much_per_share(capital_invested,starting_date,test_portfolio_1)
    historical_prices=get_historical_prices(starting_date,test_portfolio_1)
    review_prices_1=get_historical_prices(review_date,test_portfolio_1)
    todays_prices=get_historical_prices(final_date,test_portfolio_2)
    capital_available_on_review=0
    past_value=0
    present_value=0
    for s in test_portfolio_1.index:
        capital_available_on_review+=review_prices_1[s] #starting_portfolio[s]
        past_value+=starting_portfolio[s]*historical_prices[s]
    reviewed_portfolio=how_much_per_share(capital_available_on_review,review_date,test_portfolio_2)
    for s in test_portfolio_2.index:
        present_value+=reviewed_portfolio[s]*todays_prices[s]
    return (present_value-past_value)/past_value*100, past_value, capital_available_on_review

In [50]:
#annual compound is considered
#How do I get the yearly compound?
#the main issue is that the count of each stock is different, depending on when it went public
#it is actually not a problem, because, even if the count is different, the distance between 
#the two dates will be the same
def get_yearly_portfolio_growth_with_rebalancing(capital_invested,starting_date,review_date,final_date,test_portfolio_1,test_portfolio_2,months_list):
    compounded_growth=portfolio_growth_with_rebalancing(capital_invested,starting_date,review_date,final_date,test_portfolio_1,test_portfolio_2)[0]
    starting_day=get_corresponding_day(starting_date,'MMM',months_list)
    final_day=get_corresponding_day(final_date,'MMM',months_list)
    #get the historical prices of a random stock to see how many days have been recorded in total
    #historical_p=r.get("https://financialmodelingprep.com/api/v3/historical-price-full/"+'MMM'+"?serietype=line")
    #historical_p=historical_p.json()
    #today=len(historical_p['historical'])
    #now I calculated the number of years between today and the starting day
    years=(starting_day-final_day)/365
    return ((1+compounded_growth/100)**(1/years)-1)*100

In [58]:
def get_portfolio_value(current_date,portfolio_weights,portfolio,months_list):
    stock_prices=get_historical_prices(current_date,portfolio,months_list)
    portfolio_value=0
    for s in portfolio.index:
        portfolio_value+=portfolio_weights[s]*stock_prices[s]
    return(portfolio_value)

In [ ]:
get_portfolio_value('2010-02-11',weights,test_portfolio_2009)

In [ ]:
test_portfolio_2009.index

In [ ]:
test_portfolio_2012.index

In [ ]:
int((A-B)/30)

### Let's calculate the time-weighted return of a portfolio

In [ ]:
#I should consider reducing the deposit_interval to 20 days. Because the weekend do not count as trading days, which means that 30 trading 
#days are more than a month.
#According to Wikipedia, the average trading days are 21 per month
#Another thing I should look into is how the portfolio growth would change if the constraints are relaxed a bit (e.g. PExPB<23, Current ratio>1.8)


In [ ]:
date="".join(date)

In [ ]:
date='2012-12-31'

In [ ]:
c=list(date)
#1,2,3,4 is the year
#6,7 are the month

In [ ]:
a=c[5]
b=c[6]
a+b

In [32]:
#I should use 1/bond yield as a threshold for deciding whether to sell a stock or not at a certain point in time
AA_bond_yield=pd.read_excel(r'C:\Users\filip\Downloads\hqm_qh_pars.xls')
AA_bond_yield=AA_bond_yield.drop(labels='Unnamed: 1',axis=1)
AA_bond_yield=AA_bond_yield.iloc[5:,:]
AA_bond_yield=AA_bond_yield.drop(labels=['Unnamed: 2','Unnamed: 3','Unnamed: 5'],axis=1)
AA_bond_yield=AA_bond_yield.rename(columns={"The Treasury High Quality Market (HQM) Corporate Bond Yield Curve": "Period", "Unnamed: 4": "10y Yield"})
AA_bond_yield=AA_bond_yield.set_index('Period')

In [24]:
def get_maximum_PE(date,AA_bond_yield):
    date=list(date)
    temp1=date[5]
    temp2=date[6]
    month=temp1+temp2
    if month=='01':
        m='Jan'
    elif month=='02':
        m='Feb'
    elif month=='02':
        m='Feb'
    elif month=='03':
        m='Mar'
    elif month=='04':
        m='Apr'
    elif month=='05':
        m='May'
    elif month=='06':
        m='Jun'
    elif month=='07':
        m='Jul'
    elif month=='08':
        m='Aug'
    elif month=='09':
        m='Sep'
    elif month=='10':
        m='Oct'
    elif month=='11':
        m='Nov'
    elif month=='12':
        m='Dec'
    temp1=date[0]
    temp2=date[1]
    temp3=date[2]
    temp4=date[3]
    year=temp1+temp2+temp3+temp4
    period=m+" "+year
    bond_yield=AA_bond_yield.loc[period,'10y Yield']/100
    return (1/bond_yield*0.8)
    



    
    

In [ ]:
def create_test_portfolio_2(SP500_stats):
    test_portfolio=SP500_stats[SP500_stats['PExPB']<=23]
    
    test_portfolio=test_portfolio[test_portfolio['PExPB']>0]

    test_portfolio=test_portfolio[test_portfolio['Current ratio']>1.8]

    test_portfolio=test_portfolio[test_portfolio['Liquidity check']=='ok']

    test_portfolio=test_portfolio[test_portfolio['Earnings Growth %']>=0]
    
    return test_portfolio

In [ ]:
to_be_dropped,to_be_kept,new_portfolio,new_portfolio_weights,earnings_down_1y

In [ ]:
SP500_stats_2012.loc[['ATVI',
  'A',
  'ALL',
  'CF',
  'BEN',
  'GL',
  'HFC',
  'HBAN',
  'J',
  'LEN',
  'NKE',
  'NTRS',
  'PBCT',
  'RJF',
  'TFC',
  'UAA',
  'UNM']]

In [ ]:
SP500_stats_2013.loc[['ATVI',
  'A',
  'ALL',
  'CF',
  'BEN',
  'GL',
  'HFC',
  'HBAN',
  'J',
  'LEN',
  'NKE',
  'NTRS',
  'PBCT',
  'RJF',
  'TFC',
  'UAA',
  'UNM']]

In [ ]:
portfolio_weights=how_much_per_share(200,'2012-12-31',test_portfolio_2012)
portfolio_value=get_portfolio_value('2013-12-31',portfolio_weights,test_portfolio_2012)
earnings_down_last_y=[]
get_rebalanced_portfolio(AA_bond_yield,200,portfolio_weights,portfolio_value,'2013-12-31',test_portfolio_2012,test_portfolio_2013,SP500_stats_2013,earnings_down_last_y)

In [66]:
def get_rebalanced_portfolio(AA_bond_yield,recurring_investment,portfolio_weights,portfolio_value,review_date,filtering_1,filtering_2,DF_2,earnings_down_last_y,months_list):
    #to be dropped and #to_be_added are only needed if the final goal is to automatize the trading itself.
    #in that way, degiro will know which ones it has to drop, and which ones it has to buy.
    to_be_dropped=[]
    to_be_added=[]
    prices_on_review_date=get_historical_prices(review_date,filtering_1,months_list)
    #Lets find out what new stocks have been selected in the past year
    for i in filtering_1.index: 
        if i not in filtering_2.index:
            to_be_dropped.append(i)
    for j in filtering_2.index:
        if j not in filtering_1.index:
             to_be_added.append(j)
    to_be_dropped_DF_2=DF_2.loc[to_be_dropped]
    #Lets see how many constraints the stocks from filtering_1 have infringed and by how much
    #if the violation is within an acceptable range the stock will ke kept anyway
    #I also need to change Liquidity check into a numeric figure instead of an object
    #Also, a figure for the Liquidity check needs to be determined 
    earnings_down_1y=[]
    to_be_kept=[]
    for s in to_be_dropped:
        warning=0
        if to_be_dropped_DF_2.loc[s,'PExPB']>22:
            if to_be_dropped_DF_2.loc[s,'PE']<=get_maximum_PE(review_date,AA_bond_yield):
                warning+=0
            else:
                warning+=1
        if to_be_dropped_DF_2.loc[s,'Current ratio']<2:
            if to_be_dropped_DF_2.loc[s,'Current ratio']>1.8:
                warning+=0
            else:
                warning+=1
        #if to_be_dropped_DF_2.loc[s,'Liquidity check']<0:
        #    if abs(to_be_dropped_DF_2.loc[s,'Liquidity check'])<1000: 
        
        if s in earnings_down_last_y:
            if to_be_dropped_DF_2.loc[s,'Earnings Growth %']<0:
                warning+=1
        #Now I check if warning is still null or if one of the "flags" were activated. In that case, the stock will we dropped
        if warning==0:
            to_be_kept.append(s)
            if to_be_dropped_DF_2.loc[s,'Earnings Growth %']<0:
                earnings_down_1y.append(s)
    new_stocks=list(filtering_2.index)+to_be_kept
    new_portfolio=DF_2.loc[new_stocks]
    new_portfolio_weights=how_much_per_share(portfolio_value+recurring_investment,review_date,new_portfolio,months_list)
    return to_be_dropped,to_be_kept,new_portfolio,new_portfolio_weights,earnings_down_1y

In [98]:
portfolio_weights=how_much_per_share(200,'2012-12-31',test_portfolio_2012,months_list)
portfolio_value=get_portfolio_value('2014-01-31',portfolio_weights,test_portfolio_2012,months_list)
earnings_down_last_y=[]
rebalanced_p=get_rebalanced_portfolio(AA_bond_yield,200,portfolio_weights,portfolio_value,'2014-01-31',test_portfolio_2012,test_portfolio_2013,SP500_stats_2013,earnings_down_last_y,months_list)

In [108]:
#for now, the deposit interval is montlhy. In an even more elaborated version the option may be given to the reader to decide whether he wants to
#deposit money monthly or every n months.
def get_time_weighted_return(AA_bond_yield,recurring_investment,starting_date,final_date,portfolios,DFs,months_list):
    #the deposit interval will now be a number e.g. 1, which will indicate that the deposit occurs every month
    twr=0
    #I will need this later to select the proper portfolio/dataframe from the input list
    DF_count=0
    portfolio=portfolios[0]
    stock_list=portfolio.index
    starting_day=get_corresponding_day(starting_date,'MMM',months_list)
    final_day=get_corresponding_day(final_date,'MMM',months_list)
    #First I determine the number of trading days that comes the closest to having regular monthly intervals
    #This is necessary because a trading month is not always the same and it is not 30 days
    k_values=[]
    rests_list=[]
    for k in range(20,30,1):
        k_values.append(k)
        rests_list.append((starting_day-final_day)%k)
    rests_dic=dict(zip(k_values,rests_list))
    #This is the value that comes the closest to having real-month intervals, and the following is the number 
    #of regular intervals
    interval_width_in_days=min(rests_dic, key=rests_dic.get)
    number_of_intervals=int((starting_day-final_day)/interval_width_in_days)
    #now that I know the number of days in each monthly interval, and the approximated number of intervals
    #I can find the approximated number of days between the starting date and the new final date
    days_interval=interval_width_in_days*number_of_intervals
    #Now we can find the number of portfolio reviews that need to be performed
    starting_date_list=list(starting_date)
    new_final_date=get_corresponding_date(starting_date,days_interval,'MMM',months_list)
    new_final_date_list=list(new_final_date)
    starting_year=int("".join(starting_date_list[:4]))
    final_year=int("".join(new_final_date_list[:4]))
    #Probably I should make a distinction between numbers that are even/not even 
    number_of_years=final_year-starting_year
    number_of_reviews=number_of_years-1
    scheduled_reviews=[]
    t=1
    while number_of_reviews>0:
        days_interval_btween_reviews=days_interval/number_of_years
        i_periods_between_reviews=round(days_interval_btween_reviews/interval_width_in_days)
        scheduled_reviews.append(i_periods_between_reviews*t)
        t+=1
        number_of_reviews-=1
########  
    for i in range(number_of_intervals):
        how_much_per_stock=[]
        if i==0:
            #First I get the initial weights of my portfolio and its initial value, 
            #which in this case I do not need to calculate
            temp1=how_much_per_share(recurring_investment,starting_date,portfolio,months_list)
            portfolio_initial_value=recurring_investment
            #Then I get the value of the portfolio at the end of period 1, which is when I make the first deposit
            portfolio_value_temp_1=get_portfolio_value(get_corresponding_date(starting_date,interval_width_in_days*(i+1),'MMM',months_list),temp1,portfolio,months_list)
            #Now I have all the data to calculate the first member of the TMW formula, HP1
            twr+=1+(portfolio_value_temp_1-(portfolio_initial_value))/portfolio_initial_value
            print (twr)
            #Now I calculate how much of each stock I bought with the new deposit
            temp2=how_much_per_share(recurring_investment,get_corresponding_date(starting_date,interval_width_in_days*(i+1),'MMM',months_list),portfolio,months_list)
            #And I finally calculate the weights of the resulting portfolio (at time 1)
            for s in portfolio.index:
                how_much_per_stock.append(temp1[s]+temp2[s])
                temp3=dict(zip(stock_list,how_much_per_stock))
########                        
        elif i in scheduled_reviews:
            #When I update the portfolio regularly in the long term I should probably give a list of all the dataframes I will need as input
            #and select the correct dataframe based on that count variable, which is augmented by one every time this part of the loop runs
            #First I calculate the updated value of the time weighted return and then I proceed with the portfolio rebalancing
            portfolio_value_temp_2=get_portfolio_value(get_corresponding_date(starting_date,interval_width_in_days*(i+1),'MMM',months_list),temp3,portfolio,months_list)
            #Now I have the final value at the end of the n turn, and the initial value, which is the final value of 
            #the previous turn. I use these two to calculate the next component of twr
            twr=twr*(1+((portfolio_value_temp_2-(portfolio_value_temp_1+recurring_investment))/(recurring_investment+portfolio_value_temp_1)))
            print (twr)
            review_date=get_corresponding_date(starting_date,interval_width_in_days*(i+1),'MMM',months_list)
            if DF_count==0:
                earnings_down_last_y=[]
            rebalanced_p=get_rebalanced_portfolio(AA_bond_yield,recurring_investment,temp3,portfolio_value_temp_2,review_date,portfolios[DF_count],portfolios[DF_count+1],DFs[DF_count+1],earnings_down_last_y,months_list)
            portfolio=rebalanced_p[2]
            temp3=rebalanced_p[3]
            print (temp3)
            earnings_down_last_y=rebalanced_p[4]
            portfolio_value_temp_1=portfolio_value_temp_2
            DF_count+=1
            stock_list=portfolio.index
        
        else:
            #Now I am at the second turn. I first calculate the value of the portfolio at the time I make the second deposit, 
            #which is the end of the second turn
            portfolio_value_temp_2=get_portfolio_value(get_corresponding_date(starting_date,interval_width_in_days*(i+1),'MMM',months_list),temp3,portfolio,months_list)
            #Now I have the final value at the end of the n turn, and the initial value, which is the final value of 
            #the previous turn. I use these two to calculate the next component of twr
            twr=twr*(1+((portfolio_value_temp_2-(portfolio_value_temp_1+recurring_investment))/(recurring_investment+portfolio_value_temp_1)))
            print (twr)
            #Now I reassign the inital value of the next turn, to the final value of the present one
            portfolio_value_temp_1=portfolio_value_temp_2
            #I now calculate the new weights of the portfolio at the end of the present turn, which will be used in
            #the next one to compute its final value
            temp1=temp3
            temp2=how_much_per_share(recurring_investment,get_corresponding_date(starting_date,interval_width_in_days*(i+1),'MMM',months_list),portfolio,months_list)
            for s in portfolio.index:
                how_much_per_stock.append(temp1[s]+temp2[s])
                temp3=dict(zip(stock_list,how_much_per_stock))

    return 100*(twr-1),temp3,portfolio_value_temp_1
   
    

In [135]:
sp_historic

,Date,Open,High,Low,Close,Adj Close,Volume
0,1927-12-30,17.660000,17.660000,17.660000,17.660000,17.660000,0
1,1928-01-03,17.760000,17.760000,17.760000,17.760000,17.760000,0
2,1928-01-04,17.719999,17.719999,17.719999,17.719999,17.719999,0
3,1928-01-05,17.549999,17.549999,17.549999,17.549999,17.549999,0
4,1928-01-06,17.660000,17.660000,17.660000,17.660000,17.660000,0
...,...,...,...,...,...,...,...
23198,2020-05-11,2915.459961,2944.250000,2903.439941,2930.189941,2930.189941,4807320000
23199,2020-05-12,2939.500000,2945.820068,2869.590088,2870.120117,2870.120117,5107710000
23200,2020-05-13,2865.860107,2874.139893,2793.149902,2820.000000,2820.000000,6143130000
23201,2020-05-14,2794.540039,2852.800049,2766.639893,2852.500000,2852.500000,5641920000


In [23]:
def get_prior_date(date):
    date=list(date)
    temp1=date[8]
    temp2=date[9]
    #Does the number of days only have one digit
    if temp1=='0':
    #Is the number of days different than 01
        if temp2!='1':
            day=int(date[9])
            ndays=day-1
            date[9]=str(ndays)
        elif temp2=='1': 
        #the day numer is 01
            temp3=date[5]
            temp4=date[6]
            month=temp3+temp4
        #the month number is 01         
            if temp3=='0' and temp4=='1':
                year="".join(date[:4])
                year=int(year)
                year=year-1
                nyear=list(str(year))
                date[0]=nyear[0]
                date[1]=nyear[1]
                date[2]=nyear[2]
                date[3]=nyear[3]
                date[5]='1'
                date[6]='2'
                date[8]='3'
                date[9]='1'            
        #the month number is not 01               
            else:
                if temp3=='1' and temp4=='0':
                    m=9
                    date[5]='0'
                    date[6]='9'

                elif temp3=='0':
                    m=int(temp4)-1
                    nmonth=str(m)
                    date[6]=nmonth
                else:
                    m=int(month)-1
                    nmonth=list(str(m))
                    date[5]=nmonth[0]
                    date[6]=nmonth[1]
        #I have to go back by one month, which means that I need the number of days from the month before the present
        #one
                ndays=int(months_list[months_list['Month']==m]['NumberOfDays'])
                ndays_list=list(str(ndays))
                date[8]=ndays_list[0]
                date[9]=ndays_list[1]
                              
    elif temp1=='1' and temp2=='0':
        date[8]='0'
        date[9]='9'
                        
    else:    
        day=int(temp1+temp2)
        ndays=day-1
        ndays_list=list(str(ndays))
        date[8]=ndays_list[0]
        date[9]=ndays_list[1]
                    
    date="".join(date)
    return(date)
                

In [153]:
#for now, the deposit interval is montlhy. In an even more elaborated version the option may be given to the reader to decide whether he wants to
#deposit money monthly or every n months.
#First you need to download the csv file with historical data from Yahoo Finance
def get_time_weighted_return_SP500(recurring_investment,starting_date,final_date,historic_prices,months_list):
    #the deposit interval will now be a number e.g. 1, which will indicate that the deposit occurs every month
    twr=0
    #I will need this later to select the proper portfolio/dataframe from the input list
    starting_day=get_corresponding_day(starting_date,'MMM',months_list)
    final_day=get_corresponding_day(final_date,'MMM',months_list)
    #First I determine the number of trading days that comes the closest to having regular monthly intervals
    #This is necessary because a trading month is not always the same and it is not 30 days
    k_values=[]
    rests_list=[]
    for k in range(20,30,1):
        k_values.append(k)
        rests_list.append((starting_day-final_day)%k)
    rests_dic=dict(zip(k_values,rests_list))
    #This is the value that comes the closest to having real-month intervals, and the following is the number 
    #of regular intervals
    interval_width_in_days=min(rests_dic, key=rests_dic.get)
    number_of_intervals=int((starting_day-final_day)/interval_width_in_days)
    #now that I know the number of days in each monthly interval, and the approximated number of intervals
    #I can find the approximated number of days between the starting date and the new final date
    days_interval=interval_width_in_days*number_of_intervals
    #Now we can find the number of portfolio reviews that need to be performed
    new_final_date=get_corresponding_date(starting_date,days_interval,'MMM',months_list)
    #
    #Probably I should make a distinction between numbers that are even/not even
########  
    for i in range(number_of_intervals):
        if i==0:
            #First I get the initial weights of my portfolio and its initial value, 
            #which in this case I do not need to calculate
            date_bool=0
            while(date_bool==0):
                if len(historic_prices[historic_prices['Date']==starting_date])==1:
                    SP500_close1=float(historic_prices[historic_prices['Date']==starting_date]['Close'])
                    date_bool+=1
                else:
                    starting_date=get_prior_date(starting_date)
            temp1=recurring_investment/SP500_close1
            portfolio_initial_value=recurring_investment
            #Then I get the value of the portfolio at the end of period 1, which is when I make the first deposit
            final_date_interval=get_corresponding_date(starting_date,interval_width_in_days*(i+1),'MMM',months_list)
            date_bool=0
            while(date_bool==0):
                if len(historic_prices[historic_prices['Date']==final_date_interval])==1:
                    SP500_close2=float(historic_prices[historic_prices['Date']==final_date_interval]['Close'])
                    date_bool+=1
                else:
                    final_date_interval=get_prior_date(final_date_interval)
            portfolio_value_temp_1=temp1*SP500_close2
            #Now I have all the data to calculate the first member of the TMW formula, HP1
            twr+=1+(portfolio_value_temp_1-(portfolio_initial_value))/portfolio_initial_value
            print (twr)
            #Now I calculate how much of each stock I bought with the new deposit
            temp2=recurring_investment/SP500_close2
            #And I finally calculate the weights of the resulting portfolio (at time 1)
            temp3=temp1+temp2
        else:
            #Now I am at the second turn. I first calculate the value of the portfolio at the time I make the second deposit, 
            #which is the end of the second turn
            final_date_interval=get_corresponding_date(starting_date,interval_width_in_days*(i+1),'MMM',months_list)
            date_bool=0
            while(date_bool==0):
                if len(historic_prices[historic_prices['Date']==final_date_interval])==1:
                    SP500_close2=float(historic_prices[historic_prices['Date']==final_date_interval]['Close'])
                    date_bool+=1
                else:
                    final_date_interval=get_prior_date(final_date_interval)
            portfolio_value_temp_2=temp3*SP500_close2
            #Now I have the final value at the end of the n turn, and the initial value, which is the final value of 
            #the previous turn. I use these two to calculate the next component of twr
            twr=twr*(1+((portfolio_value_temp_2-(portfolio_value_temp_1+recurring_investment))/(recurring_investment+portfolio_value_temp_1)))
            print (twr)
            #Now I reassign the inital value of the next turn, to the final value of the present one
            portfolio_value_temp_1=portfolio_value_temp_2
            #I now calculate the new weights of the portfolio at the end of the present turn, which will be used in
            #the next one to compute its final value
            temp1=temp3
            temp2=recurring_investment/SP500_close2
            temp3=temp1+temp2

    return 100*(twr-1),temp3,portfolio_value_temp_1
   
    

In [ ]:
get_aggregated_portfolio_growth(200,'2012-12-31','2016-12-31',test_portfolio_2012)

# Let's test another portfolio

In [ ]:
SP500_stats=SP500_stats.drop(['Unnamed: 0'],axis=1)

In [ ]:
test_portfolio=SP500_stats[SP500_stats['Current PE']<7.5]

In [ ]:
test_portfolio=test_portfolio[test_portfolio['Current PE']>0]

In [ ]:
test_portfolio

In [ ]:
get_yearly_portfolio_growth(200,'2017-03-06',test_portfolio)

# Let's create a new DataFrame with the 31/12/2009 stock data (year 10)

In [ ]:
%%time
SP500_stats_2009=pd.DataFrame(index=SP500)
for s in SP500:
    SP500_stats_2009.loc[s,'Price']=get_historical_prices("2009-12-31",SP500_stats_2009)
    SP500_stats_2009.loc[s,'EPS']=get_NI_pershare(s,10)
    SP500_stats_2009.loc[s,'PE']=get_PE(s,10)
    SP500_stats_2009.loc[s,'PB']=get_PB(s,10)
    SP500_stats_2009.loc[s,'PExPB']=SP500_stats_2009.loc[s,'PB']*SP500_stats_2009.loc[s,'PE']
    SP500_stats_2009.loc[s,'Current ratio']=get_current_ratio(s,10)
    SP500_stats_2009.loc[s,'Liquidity check']=check_liquidity(s,10)
    SP500_stats_2009.loc[s,'Revenues']=get_revenues(s,10)
    SP500_stats_2009.loc[s,'Earnings Growth']=get_EPS_growth(s,10,1)
    past_value=SP500_stats_2009.loc[s,'EPS']-SP500_stats_2009.loc[s,'Earnings Growth']
    present_value=SP500_stats_2009.loc[s,'EPS']
    if (past_value):
        if (present_value):
                SP500_stats_2009.loc[s,'Earnings Growth %']=(present_value-past_value)/past_value*100
    SP500_stats_2009.loc[s,'Earnings Growth 3y']=get_EPS_growth(s,10,3)
    past_value=SP500_stats_2009.loc[s,'EPS']-SP500_stats_2009.loc[s,'Earnings Growth 3y']
    present_value=SP500_stats_2009.loc[s,'EPS']
    if (past_value):
        if (present_value):
                SP500_stats_2009.loc[s,'Earnings Growth 3y %']=(present_value-past_value)/past_value*100
    SP500_stats_2009.loc[s,'Earnings Growth 5y']=get_EPS_growth(s,10,5)
    past_value=SP500_stats_2009.loc[s,'EPS']-SP500_stats_2009.loc[s,'Earnings Growth 5y']
    present_value=SP500_stats_2009.loc[s,'EPS']
    if (past_value):
        if (present_value):
                SP500_stats_2009.loc[s,'Earnings Growth 5y %']=(present_value-past_value)/past_value*100
    SP500_stats_2009.loc[s,'Earnings Growth 10y']=get_EPS_growth(s,10,10)
    past_value=SP500_stats_2009.loc[s,'EPS']-SP500_stats_2009.loc[s,'Earnings Growth 10y']
    present_value=SP500_stats_2009.loc[s,'EPS']
    if (past_value):
        if (present_value):
                SP500_stats_2009.loc[s,'Earnings Growth 10y %']=(present_value-past_value)/past_value*100
    SP500_stats_2009.loc[s,'Payout ratio']=get_payout_ratio(s,10)

In [ ]:
SP500_stats_2009.to_csv("SP500_stats_2009.csv")

In [ ]:
SP500_stats_2009=SP500_stats_2009[SP500_stats_2009['Price']!='no data']

In [ ]:
SP500_stats_2009=SP500_stats_2009.drop(labels=['Earnings Growth','Earnings Growth 3y','Earnings Growth 5y','Earnings Growth 10y'], axis=1)

In [ ]:
SP500_stats_2009

In [ ]:
test_portfolio_2009=SP500_stats_2009[SP500_stats_2009['PExPB']<=22]

In [ ]:
test_portfolio_2009

In [ ]:
test_portfolio_2009=test_portfolio_2009[test_portfolio_2009['Current ratio']>2]

test_portfolio_2009=test_portfolio_2009[test_portfolio_2009['Liquidity check']=='ok']


In [ ]:
len(test_portfolio_2009['Price'])

In [ ]:
test_portfolio_2009=test_portfolio_2009[test_portfolio_2009['Earnings Growth %']>=0]

In [ ]:
len(test_portfolio_2009['Price'])

In [ ]:
get_aggregated_portfolio_growth(200,'2009-12-31','2019-12-31',test_portfolio_2009)

### This portfolio has been built by applying three simple rules (perhaps one rule on the dividends could be added, by exploiting the dividend growth information available through the API) and has delivered excellent results in the ten years between 2009 and 2019


In [ ]:
get_yearly_portfolio_growth(200,'2009-12-31','2019-12-31',test_portfolio_2009)

In [ ]:
for s in test_portfolio_2009.index:
    test_portfolio_2009.loc[s,'Sector']=get_company_sector(s)    

In [ ]:
test_portfolio_2009

# Let's create a new DataFrame with the prices from 31-12-2010 (year 9)

In [ ]:
def create_stats_DF(date,year,SP500):
    SP500_stats=pd.DataFrame(index=SP500)
    historical_prices=get_historical_prices(date,SP500_stats)
    for s in SP500:
        SP500_stats.loc[s,'Price']=historical_prices[s]
        SP500_stats.loc[s,'EPS']=get_NI_pershare(s,year)
        SP500_stats.loc[s,'PE']=get_PE(s,year)
        SP500_stats.loc[s,'PB']=get_PB(s,year)
        SP500_stats.loc[s,'PExPB']=SP500_stats.loc[s,'PB']*SP500_stats.loc[s,'PE']
        SP500_stats.loc[s,'Current ratio']=get_current_ratio(s,year)
        SP500_stats.loc[s,'Liquidity check']=check_liquidity(s,year)
        SP500_stats.loc[s,'Revenues']=get_revenues(s,year)
        SP500_stats.loc[s,'Earnings Growth']=get_EPS_growth(s,year,1)
        past_value=SP500_stats.loc[s,'EPS']-SP500_stats.loc[s,'Earnings Growth']
        present_value=SP500_stats.loc[s,'EPS']
        if (past_value):
            if (present_value):
                    SP500_stats.loc[s,'Earnings Growth %']=(present_value-past_value)/past_value*100
        SP500_stats.loc[s,'Earnings Growth 3y']=get_EPS_growth(s,year,3)
        past_value=SP500_stats.loc[s,'EPS']-SP500_stats.loc[s,'Earnings Growth 3y']
        present_value=SP500_stats.loc[s,'EPS']
        if (past_value):
            if (present_value):
                    SP500_stats.loc[s,'Earnings Growth 3y %']=(present_value-past_value)/past_value*100
        SP500_stats.loc[s,'Earnings Growth 5y']=get_EPS_growth(s,year,5)
        past_value=SP500_stats.loc[s,'EPS']-SP500_stats.loc[s,'Earnings Growth 5y']
        present_value=SP500_stats.loc[s,'EPS']
        if (past_value):
            if (present_value):
                    SP500_stats.loc[s,'Earnings Growth 5y %']=(present_value-past_value)/past_value*100
        SP500_stats.loc[s,'Earnings Growth 10y']=get_EPS_growth(s,year,10)
        past_value=SP500_stats.loc[s,'EPS']-SP500_stats.loc[s,'Earnings Growth 10y']
        present_value=SP500_stats.loc[s,'EPS']
        if (past_value):
            if (present_value):
                SP500_stats.loc[s,'Earnings Growth 10y %']=(present_value-past_value)/past_value*100
        SP500_stats.loc[s,'Payout ratio']=get_payout_ratio(s,year)
    SP500_stats=SP500_stats[SP500_stats['Price']!='no data']  
    SP500_stats=SP500_stats.drop(labels=['Earnings Growth','Earnings Growth 3y','Earnings Growth 5y','Earnings Growth 10y'], axis=1)
    return SP500_stats

In [ ]:
%%time
SP500_stats_2010=create_stats_DF('2010-12-31',9,SP500)

In [ ]:
SP500_stats_2010.to_csv('SP500_stats_2010.csv')

In [415]:
def create_test_portfolio(SP500_stats):
    
    test_portfolio=SP500_stats[SP500_stats['Revenues']>500000000]
    
    test_portfolio=test_portfolio[test_portfolio['PExPB']<=22.5]
    
    test_portfolio=test_portfolio[test_portfolio['PExPB']>0]

    test_portfolio=test_portfolio[test_portfolio['Current ratio']>1.5]

    test_portfolio=test_portfolio[test_portfolio['Liquidity check']=='ok']

    test_portfolio=test_portfolio[test_portfolio['Earnings Growth %']>=0]
    
    test_portfolio=test_portfolio[test_portfolio['Earnings Growth 3y %']>=0]
    
    test_portfolio=test_portfolio[test_portfolio['Earnings Growth 5y %']>=0]
    
    test_portfolio=test_portfolio[test_portfolio['Earnings Growth 10y %']>=0]
    
    test_portfolio=test_portfolio.set_index('Unnamed: 0')
    
    for s in test_portfolio.index:
        test_portfolio.loc[s,'Company Name']=get_company_name(s)
        test_portfolio.loc[s,'Company Industry']=get_company_industry(s)
    
    return test_portfolio

In [ ]:
test_portfolio_2010=create_test_portfolio(SP500_stats_2010)

In [ ]:
updated_portfolio_with_rebalancing(200,'2009-12-31','2010-12-31',test_portfolio_2009,test_portfolio_2010)

### What happened to the "good" stocks from 2009

In [ ]:
SP500_stats_2009=pd.read_csv("SP500_stats_2009.csv")

In [ ]:
SP500_stats_2010=pd.read_csv("SP500_stats_2010.csv")

In [ ]:
test_portfolio_2009=create_test_portfolio(SP500_stats_2009)

In [ ]:
test_portfolio_2009=test_portfolio_2009.set_index('Symbol')

In [ ]:
test_portfolio_2010=create_test_portfolio(SP500_stats_2010)

In [ ]:
test_portfolio_2010=test_portfolio_2010.set_index('Symbol')

In [ ]:
SP500_stats_2009=SP500_stats_2009.set_index('Symbol')
SP500_stats_2010=SP500_stats_2010.set_index('Symbol')

In [ ]:
SP500_stats_2009.loc[test_portfolio_2009['Symbol'],:]

In [ ]:
SP500_stats_2010.loc[test_portfolio_2009['Symbol'],:]

In [ ]:
get_aggregated_portfolio_growth(200,'2009-12-31','2019-12-31',test_portfolio_2009)

In [ ]:
get_yearly_portfolio_growth(200,'2009-12-31','2019-12-31',test_portfolio_2009)

In [ ]:
get_aggregated_portfolio_growth(200,'2010-12-31','2019-12-31',test_portfolio_2010)

In [ ]:
get_yearly_portfolio_growth(200,'2010-12-31','2019-12-31',test_portfolio_2010)

In [ ]:
get_SP500_growth('2009-12-31','2019-12-31',sp_historic)

In [ ]:
get_SP500_yearly_growth('2009-12-31','2019-12-31',sp_historic)

In [ ]:
updated_portfolio_with_rebalancing(200,'2009-12-31','2010-12-31',test_portfolio_2009,test_portfolio_2010)

In [ ]:
portfolio_growth_with_rebalancing(200,'2009-12-31','2010-12-31','2019-12-31',test_portfolio_2009,test_portfolio_2010)

In [ ]:
get_yearly_portfolio_growth_with_rebalancing(200,'2009-12-31','2010-12-31','2019-12-31',test_portfolio_2009,test_portfolio_2010)
#it is normal that this is less than the gain without rebalancing. I kept the stocks from 2009 for 1 year,
#and then I bought the stocks from 2010 with less money than before (the stocks I bought in 2009 have
#dropped in value in 2010)

### Create DF for every year

In [ ]:
dates=['2011-12-31','2012-12-31','2013-12-31','2014-12-31','2015-12-31'
       '2016-12-31','2017-12-31','2018-12-31','2019-12-31']
years=[8,7,6,5,4,3,2,1,0]
DF_list=[]
for i,j in zip(dates,years):
    DF_list.append(create_stats_DF(i,j,SP500))
    print ('done')

In [ ]:
#Something went terribly wrong
DF_list[9]

In [ ]:
def create_stats_DF(date,year,SP500):
    SP500_stats=pd.DataFrame(index=SP500)
    historical_prices=get_historical_prices(date,SP500_stats)
    for s in SP500:
        SP500_stats.loc[s,'Price']=historical_prices[s]
        SP500_stats.loc[s,'EPS']=get_NI_pershare(s,year)
        SP500_stats.loc[s,'PE']=get_PE(s,year)
        SP500_stats.loc[s,'PB']=get_PB(s,year)
        SP500_stats.loc[s,'PExPB']=SP500_stats.loc[s,'PB']*SP500_stats.loc[s,'PE']
        SP500_stats.loc[s,'Current ratio']=get_current_ratio(s,year)
        SP500_stats.loc[s,'Liquidity check']=check_liquidity(s,year)
        SP500_stats.loc[s,'Revenues']=get_revenues(s,year)
        SP500_stats.loc[s,'Earnings Growth']=get_EPS_growth(s,year,1)
        past_value=SP500_stats.loc[s,'EPS']-SP500_stats.loc[s,'Earnings Growth']
        present_value=SP500_stats.loc[s,'EPS']
        if (past_value):
            if (present_value):
                    SP500_stats.loc[s,'Earnings Growth %']=(present_value-past_value)/past_value*100
        SP500_stats.loc[s,'Earnings Growth 3y']=get_EPS_growth(s,year,3)
        past_value=SP500_stats.loc[s,'EPS']-SP500_stats.loc[s,'Earnings Growth 3y']
        present_value=SP500_stats.loc[s,'EPS']
        if (past_value):
            if (present_value):
                    SP500_stats.loc[s,'Earnings Growth 3y %']=(present_value-past_value)/past_value*100
        SP500_stats.loc[s,'Earnings Growth 5y']=get_EPS_growth(s,year,5)
        past_value=SP500_stats.loc[s,'EPS']-SP500_stats.loc[s,'Earnings Growth 5y']
        present_value=SP500_stats.loc[s,'EPS']
        if (past_value):
            if (present_value):
                    SP500_stats.loc[s,'Earnings Growth 5y %']=(present_value-past_value)/past_value*100
        SP500_stats.loc[s,'Earnings Growth 10y']=get_EPS_growth(s,year,10)
        past_value=SP500_stats.loc[s,'EPS']-SP500_stats.loc[s,'Earnings Growth 10y']
        present_value=SP500_stats.loc[s,'EPS']
        if (past_value):
            if (present_value):
                SP500_stats.loc[s,'Earnings Growth 10y %']=(present_value-past_value)/past_value*100
        SP500_stats.loc[s,'Payout ratio']=get_payout_ratio(s,year)
    SP500_stats=SP500_stats[SP500_stats['Price']!='no data']  
    SP500_stats=SP500_stats.drop(labels=['Earnings Growth','Earnings Growth 3y','Earnings Growth 5y','Earnings Growth 10y'], axis=1)
    return SP500_stats

### Create DF for 2011

In [ ]:
dates='2011-12-31'
years=[8]

#create_stats_DF(i,j,SP500)

In [ ]:
SP500_stats=pd.DataFrame(index=SP500)
historical_prices=get_historical_prices(dates,SP500_stats)

In [ ]:
for s in SP500:
    SP500_stats.loc[s,'Price']=historical_prices[s]
#In the end I should probably remove the stocks that don't have any price information
#as that likely indicates that the IPO hadnt taken place yet

In [ ]:
for s in SP500:
    SP500_stats.loc[s,'EPS']=get_NI_pershare(s,8)

In [ ]:
for s in SP500:    
    SP500_stats.loc[s,'PE']=get_PE(s,8)
    SP500_stats.loc[s,'PB']=get_PB(s,8)
    SP500_stats.loc[s,'PExPB']=SP500_stats.loc[s,'PB']*SP500_stats.loc[s,'PE']
    SP500_stats.loc[s,'Current ratio']=get_current_ratio(s,8)
    SP500_stats.loc[s,'Liquidity check']=check_liquidity(s,8)
    SP500_stats.loc[s,'Revenues']=get_revenues(s,8)

In [ ]:
for s in SP500_stats.index:    
    SP500_stats.loc[s,'Earnings Growth %']=get_EPS_growth(s,8,1)
    
    SP500_stats.loc[s,'Earnings Growth 3y %']=get_EPS_growth(s,8,3)
    
    SP500_stats.loc[s,'Earnings Growth 5y %']=get_EPS_growth(s,8,5)
    
    SP500_stats.loc[s,'Earnings Growth 10y %']=get_EPS_growth(s,8,10)
    
    #SP500_stats.loc[s,'Payout ratio']=get_payout_ratio(s,8)

In [ ]:
SP500_stats=SP500_stats[SP500_stats['Price']!='no data']  

In [ ]:
for s in SP500_stats.index:
    SP500_stats.loc[s,'Net Current Assets PS']=get_net_current_assets(s,8)

In [ ]:
SP500_stats.to_csv('SP500_stats_2011.csv')

### Create DF for 2012

In [ ]:
SP500_stats_2012=pd.DataFrame(index=SP500)

In [ ]:
historical_prices=get_historical_prices('2012-12-31',SP500_stats_2012)

In [ ]:
for s in SP500:
    SP500_stats_2012.loc[s,'Price']=historical_prices[s]

In [ ]:
for s in SP500:
    SP500_stats_2012.loc[s,'EPS']=get_NI_pershare(s,7)
    SP500_stats_2012.loc[s,'PE']=get_PE(s,7)
    SP500_stats_2012.loc[s,'PB']=get_PB(s,7)
    SP500_stats_2012.loc[s,'PExPB']=SP500_stats_2012.loc[s,'PB']*SP500_stats_2012.loc[s,'PE']
    SP500_stats_2012.loc[s,'Current ratio']=get_current_ratio(s,7)
    SP500_stats_2012.loc[s,'Liquidity check']=check_liquidity(s,7)
    SP500_stats_2012.loc[s,'Revenues']=get_revenues(s,7)

In [ ]:
for s in SP500_stats_2012.index:    
    SP500_stats_2012.loc[s,'Earnings Growth %']=get_EPS_growth(s,7,1)
    
    SP500_stats_2012.loc[s,'Earnings Growth 3y %']=get_EPS_growth(s,7,3)
    
    SP500_stats_2012.loc[s,'Earnings Growth 5y %']=get_EPS_growth(s,7,5)
    
    SP500_stats_2012.loc[s,'Earnings Growth 10y %']=get_EPS_growth(s,7,10)
    
    #SP500_stats_2012.loc[s,'Payout ratio']=get_payout_ratio(s,7)
    
SP500_stats_2012=SP500_stats_2012[SP500_stats_2012['Price']!='no data']  


In [ ]:
for s in SP500_stats_2012.index:
    SP500_stats_2012.loc[s,'Net Current Assets PS']=get_net_current_assets(s,7)

In [ ]:
SP500_stats_2012.to_csv('SP500_stats_2012.csv')

### Create DF 2013

In [ ]:
SP500_stats_2013=pd.DataFrame(index=SP500)

In [ ]:
historical_prices=get_historical_prices('2013-12-31',SP500_stats_2013)

In [ ]:
for s in SP500:
    SP500_stats_2013.loc[s,'Price']=historical_prices[s]

In [ ]:
for s in SP500:
    SP500_stats_2013.loc[s,'EPS']=get_NI_pershare(s,6)
    SP500_stats_2013.loc[s,'PE']=get_PE(s,6)
    SP500_stats_2013.loc[s,'PB']=get_PB(s,6)
    SP500_stats_2013.loc[s,'PExPB']=SP500_stats_2013.loc[s,'PB']*SP500_stats_2013.loc[s,'PE']
    SP500_stats_2013.loc[s,'Current ratio']=get_current_ratio(s,6)
    SP500_stats_2013.loc[s,'Liquidity check']=check_liquidity(s,6)
    SP500_stats_2013.loc[s,'Revenues']=get_revenues(s,6)

In [ ]:
for s in SP500_stats_2013.index:    
    SP500_stats_2013.loc[s,'Earnings Growth %']=get_EPS_growth(s,6,1)
    
    SP500_stats_2013.loc[s,'Earnings Growth 3y %']=get_EPS_growth(s,6,3)
    
    SP500_stats_2013.loc[s,'Earnings Growth 5y %']=get_EPS_growth(s,6,5)
    
    SP500_stats_2013.loc[s,'Earnings Growth 10y %']=get_EPS_growth(s,6,10)
    
    #SP500_stats_2013.loc[s,'Payout ratio']=get_payout_ratio(s,6)
    
SP500_stats_2013=SP500_stats_2013[SP500_stats_2013['Price']!='no data']  


In [ ]:
for s in SP500_stats_2013.index:
    SP500_stats_2013.loc[s,'Net Current Assets PS']=get_net_current_assets(s,6)

In [ ]:
SP500_stats_2013.to_csv('SP500_stats_2013.csv')

In [ ]:
SP500_stats_2013

### Create DF 2014

In [ ]:
SP500_stats_2014=pd.DataFrame(index=SP500)

In [ ]:
historical_prices=get_historical_prices('2014-12-31',SP500_stats_2014)

In [ ]:
for s in SP500:
    SP500_stats_2014.loc[s,'Price']=historical_prices[s]

In [ ]:
for s in SP500:
    SP500_stats_2014.loc[s,'EPS']=get_NI_pershare(s,5)
    SP500_stats_2014.loc[s,'PE']=get_PE(s,5)
    SP500_stats_2014.loc[s,'PB']=get_PB(s,5)
    SP500_stats_2014.loc[s,'PExPB']=SP500_stats_2014.loc[s,'PB']*SP500_stats_2014.loc[s,'PE']
    SP500_stats_2014.loc[s,'Current ratio']=get_current_ratio(s,5)
    SP500_stats_2014.loc[s,'Liquidity check']=check_liquidity(s,5)
    SP500_stats_2014.loc[s,'Revenues']=get_revenues(s,5)

In [ ]:
for s in SP500_stats_2014.index:    
    SP500_stats_2014.loc[s,'Earnings Growth %']=get_EPS_growth(s,5,1)
    
    SP500_stats_2014.loc[s,'Earnings Growth 3y %']=get_EPS_growth(s,5,3)
    
    SP500_stats_2014.loc[s,'Earnings Growth 5y %']=get_EPS_growth(s,5,5)
    
    SP500_stats_2014.loc[s,'Earnings Growth 10y %']=get_EPS_growth(s,5,10)
    
    #SP500_stats_2014.loc[s,'Payout ratio']=get_payout_ratio(s,5)
    
SP500_stats_2014=SP500_stats_2014[SP500_stats_2014['Price']!='no data']  


In [ ]:
for s in SP500_stats_2014.index:
    SP500_stats_2014.loc[s,'Net Current Assets PS']=get_net_current_assets(s,5)

In [ ]:
SP500_stats_2014.to_csv('SP500_stats_2014.csv')

### Create DF 2015

In [ ]:
SP500_stats_2015=pd.DataFrame(index=SP500)

In [ ]:
historical_prices=get_historical_prices('2015-12-31',SP500_stats_2015)

In [ ]:
for s in SP500:
    SP500_stats_2015.loc[s,'Price']=historical_prices[s]

In [ ]:
for s in SP500:
    SP500_stats_2015.loc[s,'EPS']=get_NI_pershare(s,4)
    SP500_stats_2015.loc[s,'PE']=get_PE(s,4)
    SP500_stats_2015.loc[s,'PB']=get_PB(s,4)
    SP500_stats_2015.loc[s,'PExPB']=SP500_stats_2015.loc[s,'PB']*SP500_stats_2015.loc[s,'PE']
    SP500_stats_2015.loc[s,'Current ratio']=get_current_ratio(s,4)
    SP500_stats_2015.loc[s,'Liquidity check']=check_liquidity(s,4)
    SP500_stats_2015.loc[s,'Revenues']=get_revenues(s,4)

In [ ]:
for s in SP500_stats_2015.index:    
    SP500_stats_2015.loc[s,'Earnings Growth %']=get_EPS_growth(s,4,1)
    
    SP500_stats_2015.loc[s,'Earnings Growth 3y %']=get_EPS_growth(s,4,3)
    
    SP500_stats_2015.loc[s,'Earnings Growth 5y %']=get_EPS_growth(s,4,5)
    
    SP500_stats_2015.loc[s,'Earnings Growth 10y %']=get_EPS_growth(s,4,10)
    
    #SP500_stats_2015.loc[s,'Payout ratio']=get_payout_ratio(s,4)
    
SP500_stats_2015=SP500_stats_2015[SP500_stats_2015['Price']!='no data']  


In [ ]:
for s in SP500_stats_2015.index:
    SP500_stats_2015.loc[s,'Net Current Assets PS']=get_net_current_assets(s,4)

In [ ]:
SP500_stats_2015.to_csv('SP500_stats_2015.csv')

### Create DF 2016

In [ ]:
SP500_stats_2016=pd.DataFrame(index=SP500)

In [ ]:
historical_prices=get_historical_prices('2016-12-31',SP500_stats_2016)

In [ ]:
for s in SP500:
    SP500_stats_2016.loc[s,'Price']=historical_prices[s]

In [ ]:
for s in SP500:
    SP500_stats_2016.loc[s,'EPS']=get_NI_pershare(s,3)
    SP500_stats_2016.loc[s,'PE']=get_PE(s,3)
    SP500_stats_2016.loc[s,'PB']=get_PB(s,3)
    SP500_stats_2016.loc[s,'PExPB']=SP500_stats_2016.loc[s,'PB']*SP500_stats_2016.loc[s,'PE']
    SP500_stats_2016.loc[s,'Current ratio']=get_current_ratio(s,3)
    SP500_stats_2016.loc[s,'Liquidity check']=check_liquidity(s,3)
    SP500_stats_2016.loc[s,'Revenues']=get_revenues(s,3)

In [ ]:
for s in SP500_stats_2016.index:    
    SP500_stats_2016.loc[s,'Earnings Growth %']=get_EPS_growth(s,3,1)
    
    SP500_stats_2016.loc[s,'Earnings Growth 3y %']=get_EPS_growth(s,3,3)
    
    SP500_stats_2016.loc[s,'Earnings Growth 5y %']=get_EPS_growth(s,3,5)
    
    SP500_stats_2016.loc[s,'Earnings Growth 10y %']=get_EPS_growth(s,3,10)
    
    #SP500_stats_2016.loc[s,'Payout ratio']=get_payout_ratio(s,3)
    
SP500_stats_2016=SP500_stats_2016[SP500_stats_2016['Price']!='no data']  


In [ ]:
for s in SP500_stats_2016.index:
    SP500_stats_2016.loc[s,'Net Current Assets PS']=get_net_current_assets(s,3)

In [ ]:
SP500_stats_2016.to_csv('SP500_stats_2016.csv')

### Create DF 2017

In [ ]:
SP500_stats_2017=create_stats_DF('2017-12-31',2,SP500)

In [ ]:
for s in SP500_stats_2017.index:    
    SP500_stats_2017.loc[s,'Earnings Growth %']=get_EPS_growth(s,2,1)
    
    SP500_stats_2017.loc[s,'Earnings Growth 3y %']=get_EPS_growth(s,2,3)
    
    SP500_stats_2017.loc[s,'Earnings Growth 5y %']=get_EPS_growth(s,2,5)
    
    SP500_stats_2017.loc[s,'Earnings Growth 10y %']=get_EPS_growth(s,2,10)
    
    #SP500_stats_2017.loc[s,'Payout ratio']=get_payout_ratio(s,2)
    
SP500_stats_2017=SP500_stats_2017[SP500_stats_2017['Price']!='no data'] 

In [ ]:
for s in SP500_stats_2017.index:
    SP500_stats_2017.loc[s,'Net Current Assets PS']=get_net_current_assets(s,2)

In [ ]:
SP500_stats_2017.to_csv('SP500_stats_2017.csv')

### Create DF 2018

In [ ]:
SP500_stats_2018=create_stats_DF('2018-12-31',1,SP500)

In [ ]:
for s in SP500_stats_2018.index:    
    SP500_stats_2018.loc[s,'Earnings Growth %']=get_EPS_growth(s,1,1)
    
    SP500_stats_2018.loc[s,'Earnings Growth 3y %']=get_EPS_growth(s,1,3)
    
    SP500_stats_2018.loc[s,'Earnings Growth 5y %']=get_EPS_growth(s,1,5)
    
    SP500_stats_2018.loc[s,'Earnings Growth 10y %']=get_EPS_growth(s,1,10)
    
    #SP500_stats_2018.loc[s,'Payout ratio']=get_payout_ratio(s,1)
    
SP500_stats_2018=SP500_stats_2018[SP500_stats_2018['Price']!='no data'] 

In [ ]:
for s in SP500_stats_2018.index:
    SP500_stats_2018.loc[s,'Net Current Assets PS']=get_net_current_assets(s,1)

In [ ]:
SP500_stats_2018.to_csv('SP500_stats_2018.csv')

### Let's create the test portfolio for every year

In [ ]:

test_portfolio_2012=create_test_portfolio(SP500_stats_2012)

test_portfolio_2013=create_test_portfolio(SP500_stats_2013)

test_portfolio_2014=create_test_portfolio(SP500_stats_2014)

test_portfolio_2015=create_test_portfolio(SP500_stats_2015)

test_portfolio_2016=create_test_portfolio(SP500_stats_2016)

test_portfolio_2017=create_test_portfolio(SP500_stats_2017)

test_portfolio_2018=create_test_portfolio(SP500_stats_2018)

In [ ]:
test_portfolio_2009

In [ ]:
SP500_stats_2010.loc[test_portfolio_2009.index]

In [ ]:
portfolio_growth_with_rebalancing(200,'2009-12-31','2014-12-31','2019-12-31',test_portfolio_2009,test_portfolio_2014)

In [ ]:
portfolio_growth_with_rebalancing(200,'2009-12-31','2015-12-31','2019-12-31',test_portfolio_2009,test_portfolio_2015)

In [ ]:
portfolio_growth_with_rebalancing(200,'2009-12-31','2016-12-31','2019-12-31',test_portfolio_2009,test_portfolio_2016)

In [ ]:
portfolio_growth_with_rebalancing(200,'2009-12-31','2017-12-31','2019-12-31',test_portfolio_2009,test_portfolio_2017)

In [ ]:
portfolio_growth_with_rebalancing(200,'2009-12-31','2018-12-31','2019-12-31',test_portfolio_2009,test_portfolio_2018)

In [ ]:
get_aggregated_portfolio_growth(200,'2017-12-31','2019-12-31',test_portfolio_2017)

In [ ]:
get_aggregated_portfolio_growth(200,'2009-12-31','2019-12-31',test_portfolio_2009)

In [ ]:
updated_portfolio_with_rebalancing(200,'2010-12-31','2011-12-31',test_portfolio_2010,test_portfolio_2011)

### Lets try the time weigthed return function

In [36]:
#First I need to load all the DFs and portfolios, which serve as in input for the time weighted return function
SP500_stats_2012=pd.read_csv('SP500_stats_2012.csv')
SP500_stats_2013=pd.read_csv('SP500_stats_2013.csv')
SP500_stats_2014=pd.read_csv('SP500_stats_2014.csv')
SP500_stats_2015=pd.read_csv('SP500_stats_2015.csv')
SP500_stats_2016=pd.read_csv('SP500_stats_2016.csv')
SP500_stats_2017=pd.read_csv('SP500_stats_2017.csv')
SP500_stats_2018=pd.read_csv('SP500_stats_2018.csv')

In [37]:
#Set the index of the DFs to the stock symbols
SP500_stats_2012=SP500_stats_2012.set_index('Symbol',drop=True)
SP500_stats_2013=SP500_stats_2013.set_index('Symbol',drop=True)
SP500_stats_2014=SP500_stats_2014.set_index('Symbol',drop=True)
SP500_stats_2015=SP500_stats_2015.set_index('Symbol',drop=True)
SP500_stats_2016=SP500_stats_2016.set_index('Symbol',drop=True)
SP500_stats_2017=SP500_stats_2017.set_index('Symbol',drop=True)
SP500_stats_2018=SP500_stats_2018.set_index('Symbol',drop=True)

In [38]:
DFs=[SP500_stats_2012,SP500_stats_2013,SP500_stats_2014,SP500_stats_2015,SP500_stats_2016,SP500_stats_2017,SP500_stats_2018]

In [41]:

test_portfolio_2012=create_test_portfolio(SP500_stats_2012)

test_portfolio_2013=create_test_portfolio(SP500_stats_2013)

test_portfolio_2014=create_test_portfolio(SP500_stats_2014)

test_portfolio_2015=create_test_portfolio(SP500_stats_2015)

test_portfolio_2016=create_test_portfolio(SP500_stats_2016)

test_portfolio_2017=create_test_portfolio(SP500_stats_2017)

test_portfolio_2018=create_test_portfolio(SP500_stats_2018)

In [43]:
portfolios=[test_portfolio_2012,test_portfolio_2013,test_portfolio_2014,test_portfolio_2015,test_portfolio_2016,test_portfolio_2017,test_portfolio_2018]

In [109]:
get_time_weighted_return(AA_bond_yield,200,'2012-12-31','2014-12-31',portfolios,DFs,months_list)

1.0898621602790721
1.1169794706445322
1.1292332231061422
1.1576066606468105
1.2010360105890887
1.190482701619565
1.292875154387492
1.217228202908588
1.2774080324661712
1.326744713038683
1.3732673465862315
1.4232696579758346
1.3865905149574806
{'AOS': 3.2998210743048277, 'BAC': 4.651270182945491, 'COF': 1.1033674488647072, 'CB': 0.8304954222826668, 'CINF': 1.6080242634538076, 'CMA': 1.7010649686536459, 'GLW': 4.5269480281427645, 'EW': 2.3927756622953615, 'GRMN': 1.7293845852239065, 'HP': 0.8849247565235913, 'HRL': 3.4290834315289165, 'ISRG': 0.5734489589602308, 'KEY': 6.105703414132992, 'MTB': 0.6986707520790689, 'NWSA': 4.881502228341915, 'PNC': 0.9753226785720703, 'PFG': 1.7881288860302267, 'RF': 7.660646564831561, 'STT': 1.1636859681006269, 'SIVB': 0.6941885018652497, 'TRV': 0.9585233214116262, 'ZION': 2.709870454411721, 'ATVI': 4.548089641817687, 'A': 1.8728071049119466, 'ALL': 1.5216557727409565, 'CF': 1.6874328690564646, 'DHI': 3.318090952484539, 'BEN': 1.4979576151574117, 'GL': 1

(57.18665406136716,
 {'AOS': 5.460821094225956,
  'BAC': 7.939418862107534,
  'COF': 1.776410325847532,
  'CB': 1.3429627743862045,
  'CINF': 2.7135409366840526,
  'CMA': 2.8038544656592683,
  'GLW': 7.114188987979025,
  'EW': 3.5461863934191284,
  'GRMN': 2.6971626886886595,
  'HP': 1.4561970904699202,
  'HRL': 5.591337043455059,
  'ISRG': 0.9310108867655599,
  'KEY': 10.044267740326877,
  'MTB': 1.1383096124345493,
  'NWSA': 8.055051355357689,
  'PNC': 1.6013008968096716,
  'PFG': 2.8654299688175757,
  'RF': 12.87425452464218,
  'STT': 1.9249870663687343,
  'SIVB': 1.172734942145245,
  'TRV': 1.5315390305199215,
  'ZION': 4.5521456935715445,
  'ATVI': 7.109873374891872,
  'A': 3.195234221219257,
  'ALL': 2.4096554424339374,
  'CF': 2.731032048843231,
  'DHI': 5.657589912636936,
  'BEN': 2.472945969911669,
  'GL': 2.5616471581809246,
  'HFC': 2.8722250238561555,
  'HBAN': 14.09317530956652,
  'J': 2.3145788816667205,
  'LEN': 3.298255489185293,
  'LNC': 2.644862336679924,
  'NKE': 3.4

In [ ]:
 #### I think the problem is that it has not found the review date, so it has returned 'no data'

In [118]:
get_time_weighted_return(AA_bond_yield,200,'2012-12-31','2015-12-31',portfolios,DFs,months_list)

1.0898621602790721
1.1169794706445322
1.1292332231061422
1.1576066606468105
1.2010360105890887
1.190482701619565
1.292875154387492
1.217228202908588
1.2774080324661712
1.326744713038683
1.3732673465862315
1.4232696579758346
1.3865905149574806
{'AOS': 3.2998210743048277, 'BAC': 4.651270182945491, 'COF': 1.1033674488647072, 'CB': 0.8304954222826668, 'CINF': 1.6080242634538076, 'CMA': 1.7010649686536459, 'GLW': 4.5269480281427645, 'EW': 2.3927756622953615, 'GRMN': 1.7293845852239065, 'HP': 0.8849247565235913, 'HRL': 3.4290834315289165, 'ISRG': 0.5734489589602308, 'KEY': 6.105703414132992, 'MTB': 0.6986707520790689, 'NWSA': 4.881502228341915, 'PNC': 0.9753226785720703, 'PFG': 1.7881288860302267, 'RF': 7.660646564831561, 'STT': 1.1636859681006269, 'SIVB': 0.6941885018652497, 'TRV': 0.9585233214116262, 'ZION': 2.709870454411721, 'ATVI': 4.548089641817687, 'A': 1.8728071049119466, 'ALL': 1.5216557727409565, 'CF': 1.6874328690564646, 'DHI': 3.318090952484539, 'BEN': 1.4979576151574117, 'GL': 1

(54.86682686877447,
 {'ALL': 3.664395708106314,
  'BK': 6.6912785671486255,
  'COF': 3.347570931281968,
  'CINF': 4.849227389184321,
  'CFG': 10.387370532360766,
  'CMA': 5.80342590774652,
  'GLW': 11.18007281115447,
  'GL': 4.808948668408466,
  'HAL': 6.1454162057162405,
  'HRL': 9.35452170133558,
  'LNC': 4.8386472795382565,
  'MET': 5.827043590670355,
  'MS': 7.308155676382087,
  'NOV': 4.875441833691842,
  'PBCT': 17.239306725597444,
  'PFG': 5.208031386931818,
  'RF': 27.709966752867935,
  'TRV': 2.4174361182227746,
  'WRB': 7.527899967632834,
  'ZION': 9.878488882554183,
  'AOS': 8.13760573154136,
  'BAC': 16.095582099253647,
  'CB': 2.3188804354122015,
  'EW': 3.894989922019654,
  'GRMN': 5.189331330936809,
  'HP': 4.026146778333317,
  'ISRG': 1.5013038917034254,
  'KEY': 18.878971528594505,
  'MTB': 2.158624628799464,
  'PNC': 2.877878295150207,
  'STT': 3.4706937622075102,
  'SIVB': 2.1166508244501996},
 8000.214097963231)

In [ ]:
#return to_be_dropped,to_be_kept,new_portfolio,new_portfolio_weights,earnings_down_1y

In [154]:
get_time_weighted_return_SP500(200,'2012-12-31','2015-12-31',sp_historic,months_list)

1.0504280965195785
1.0694227375706893
1.089399031878321
1.1201803631287846
1.150211480842298
1.1317426309066918
1.1968041183933713
1.1449877215197657
1.188481247323563
1.2363781347129827
1.2661778120057563
1.296012495855907
1.2498966054620342
1.31392739503272
1.3258402472493656
1.31899683269467
1.3492171937847077
1.3845420853378447
1.3498552813029534
1.403936440328601
1.3645868464304343
1.3985865182876067
1.4398081783974646
1.4436365331229029
1.416957109221399
1.4714239447857647
1.4492879956443312
1.4826145727247115
1.4823201364873435
1.4561735217006389
1.4710803790474891
1.341932039331356
1.3489226720047356
1.4579825920957015
1.474298634812763
1.4331470740614316


(43.31470740614316, 4.001682627792788, 7979.198954151517)

In [155]:
get_time_weighted_return(AA_bond_yield,200,'2012-12-31','2019-12-31',portfolios,DFs,months_list)

1.084593144400003
1.1038129908924803
1.1581795615173929
1.1558986172208052
1.2035472020062337
1.1480633784472312
1.2599299514121194
1.2490282721697932
1.2903619895155225
1.3066840978453214
1.3371655560920273
1.3562086586726803
1.4102612315213898
1.4021631031951025
{'AOS': 3.554417334467645, 'BAC': 4.984185918652202, 'COF': 1.1806491482858428, 'CB': 0.86905338608459, 'CINF': 1.8163440912532203, 'CMA': 1.8104811342123772, 'GLW': 4.475162675896232, 'EW': 2.5136856380893087, 'GRMN': 1.8588833032887575, 'HP': 0.955080693686561, 'HRL': 3.789777401209422, 'ISRG': 0.5918611206953863, 'KEY': 6.567764114508131, 'MTB': 0.7380740267290917, 'NWSA': 4.69230665403509, 'PNC': 1.038679732183323, 'PFG': 1.9239208393974199, 'RF': 8.168258088043608, 'STT': 1.2237535753723514, 'SIVB': 0.7409340229577205, 'TRV': 1.0110931174960842, 'ZION': 2.811933766940146, 'ATVI': 4.343472292558036, 'A': 1.9689458999964702, 'ALL': 1.5931274059240514, 'CF': 1.820667784177649, 'DHI': 3.6062176728182243, 'BEN': 1.59312740592

(162.55224397413213,
 {'BAC': 43.765997018850655,
  'BK': 24.909010865384936,
  'CB': 9.174484339454287,
  'CFG': 37.00112308478309,
  'DHI': 28.259723203902283,
  'HBAN': 95.24594812763861,
  'KEY': 77.13835191743688,
  'MET': 28.054863996679394,
  'MU': 30.096514965270664,
  'MS': 28.470598872530473,
  'NUE': 22.248165637047038,
  'PBCT': 75.75162861950061,
  'RF': 83.7609478154613,
  'STT': 18.536187043197078,
  'TRV': 9.32363779490074,
  'TFC': 26.221536847739454,
  'USB': 25.519825870691406,
  'LEN': 24.77296156462791,
  'ZION': 26.924627983643827},
 27041.328255459375)

In [156]:
get_time_weighted_return_SP500(200,'2012-12-31','2019-12-31',sp_historic,months_list)

1.0531275798698119
1.0620465132000254
1.1002671494792151
1.1094174376875652
1.1566481634580539
1.1030017256305975
1.186651207070882
1.1585763778711156
1.2098809354875402
1.2070902966773902
1.2494829396640652
1.2449253419604644
1.2755663875489356
1.275952064788823
1.3099236625453101
1.3127143868980629
1.3151333851680835
1.3668866151398549
1.3866666473704985
1.3595594347274949
1.4039364403286023
1.382908392704755
1.3918553145930503
1.4493371251452398
1.4597493806051172
1.4423675478720819
1.4833087229017268
1.466494707243205
1.4780359196208908
1.490579926899091
1.4873474626476832
1.491133706572678
1.4665227813438901
1.3694038731128582
1.4145801362092194
1.4574356600373743
1.4469251455756824
1.3624342102970974
1.3182325824579624
1.403571819190105
1.4340095840011247
1.446770908756535
1.4719849142450225
1.4519594904364836
1.521578575626764
1.5232683309186226
1.517813216016785
1.501314691995853
1.5298803050525787
1.586415640691999
1.5872289348870112
1.645797592257734
1.6675548828597437
1.6470

(127.18012004265007, 8.172438355937992, 26278.86388545498)

In [157]:
((1+162/100)**(1/7)-1)*100

14.751224177644984

In [158]:
((1+127/100)**(1/7)-1)*100

12.424466862842865

# Let's take a look at all listed stocks on NYSE, NASDAQ

In [219]:
NYSEandNDQSymbols=pd.read_csv(r'C:\Users\filip\Downloads\companylist.csv')
NYSEandNDQ=NYSEandNDQSymbols['Symbol']

In [220]:
NYSEandNDQ

0       VCVCU
1         TXG
2          YI
3          YQ
4        TURN
        ...  
7224      ZBH
7225      ZTS
7226      ZTO
7227      ZUO
7228     ZYME
Name: Symbol, Length: 7229, dtype: object

In [400]:
7229*0.67

4843.43

In [401]:
trial_list=NYSEandNDQ[4800:]

In [315]:
from tqdm import tqdm

In [404]:
NYSEandNDQ_stats=create_stats_DF('2020-12-31',0,trial_list,months_list)

Loaded 4521 stats from cache
['VCVCU', 'TXG', 'YI', 'YQ', 'TURN', 'ATNF', 'ATNFW', 'FLWS', 'BCOW', 'ONEM', 'FCCY', 'SRCE', 'VNET', 'TWOU', 'QFIN', 'KRKR', 'FDMT', 'FVAM', 'JOBS', 'ETNB', 'EGHT', 'NMTR', 'JFU', 'AAON', 'ABCM', 'ABCL', 'ABEO', 'ABMD', 'AXAS', 'ABST', 'ACIU', 'ACIA', 'ACTG', 'ASO', 'ACHC', 'ACAD', 'ACAM', 'ACAMU', 'ACAMW', 'ACST', 'AXDX', 'ACCP', 'XLRN', 'ACCD', 'ARAY', 'ACEV', 'ACEVU', 'ACEVW', 'ACLL', 'ACRX', 'ACER', 'ACHV', 'ACIW', 'ACAC', 'ACACU', 'ACACW', 'ACRS', 'ACMR', 'ACNB', 'STWO', 'STWOU', 'STWOW', 'ACOR', 'ATVI', 'AFIB', 'ADMS', 'ADMP', 'AHCO', 'ADAP', 'ADPT', 'ADXN', 'ADUS', 'AEY', 'ADIL', 'ADILW', 'ACET', 'ADTX', 'ADMA', 'ADBE', 'ADTN', 'ADES', 'AEIS', 'AMD', 'ADV', 'ADVWW', 'ADXS', 'ADVM', 'DWEQ', 'DWAW', 'DWUS', 'DWMC', 'DWSH', 'AEGN', 'AGLE', 'AEHR', 'AMTX', 'ARBGU', 'AERI', 'AVAV', 'ARPO', 'AIH', 'AEZS', 'AEMD', 'AFMD', 'AFYA', 'AGBA', 'AGBAR', 'AGBAU', 'AGBAW', 'AGEN', 'AGRX', 'AGYS', 'AGIO', 'AGMH', 'AGNC', 'AGNCM', 'AGNCN', 'AGNCO', 'AGNCP', 'API', 'A


















  0%|          | 0/2429 [00:00<?, ?it/s]

Found GYCin the cache



















  0%|          | 1/2429 [00:03<2:02:23,  3.02s/it]

Found OFCin the cache


...


















  2%|▏         | 37/2429 [01:51<2:00:23,  3.02s/it]

Found SZCin the cache



















  2%|▏         | 38/2429 [01:54<2:00:15,  3.02s/it]

Found CWKin the cache



















  2%|▏         | 39/2429 [01:57<2:00:23,  3.02s/it]
















  2%|▏         | 40/2429 [02:07<3:21:37,  5.06s/it]
















  2%|▏         | 41/2429 [02:18<4:32:30,  6.85s/it]
















  2%|▏         | 42/2429 [02:21<3:46:37,  5.70s/it]
















  2%|▏         | 43/2429 [02:24<3:14:29,  4.89s/it]
















  2%|▏         | 44/2429 [02:27<2:51:55,  4.33s/it]
















  2%|▏         | 45/2429 [02:30<2:36:10,  3.93s/it]
















  2%|▏         | 46/2429 [02:41<4:01:17,  6.08s/it]
















  2%|▏         | 47/2429 [02:52<4:59:31,  7.54s/it]
















  2%|▏         | 48/2429 [03:03<5:44:28,  8.68s/it]
















  2%|▏         | 49/2429 [03:15<6:18:59,  9.55s/it]
















  2%|▏         | 50/2429 [03:26<6:35:40,  9.98s/it]
















  2%|▏         | 51/2429 [03:35<6:26:11,  9.74s/it]
















  2%|▏         | 52/2429 [03:38<5:06:03,  7.73s/it]
















  2%|▏         | 53/2429 [03:41

In [416]:
Stats=pd.read_csv(r'C:\Users\filip\Downloads\Stats_cache.csv')
test_p=create_test_portfolio(Stats)

In [418]:
test_p.to_csv(r'C:\Users\filip\Downloads\Filtered_Stocks.csv')